# 03 — Forecasting Experiments, Results and Figures

This notebook runs the final thesis forecasting experiment under one fixed midnight-origin protocol. It consumes the processed `train.csv`, `validation.csv`, and `test.csv` created by Notebook 02, saves auditable experiment artifacts under `results/thesis_outputs_v3/`, and writes thesis figures to the repository-level `figures/` directory.

### Forecasting protocol
- Forecast horizon: **24 consecutive physical hourly steps**
- Forecast frequency: **once per local calendar day**
- Forecast origin / first target hour: **00:00 Europe/Berlin**
- Every model is evaluated on the **same daily forecast origins and target timestamps**.
- Observed load and historical ERA5-Land reanalysis weather must come from timestamps strictly **before** the common forecast origin.
- Realised target-period grid load and weather are never supplied as predictors.
- Future-known calendar information is allowed where the model architecture uses it.
- Train: **2015–2023**
- Validation: **2024**
- Test: **2025**
- Model/configuration selection uses validation; the 2025 period is reserved for final evaluation under the frozen specification.

### Final model set
Seasonal Naive, Ridge Regression, XGBoost, LSTM, and Temporal Fusion Transformer (TFT).

Stochastic models use seeds **42, 123, and 777**. The frozen neural lookbacks are **LSTM 168 h** and **TFT 336 h** unless the optional validation lookback study is explicitly rerun.


### v3 methodological fixes implemented in this notebook

**1. Leakage-safe target-relative tabular lags.** Ridge/XGBoost retain the final observed state at `origin - 1 h`, while lags of 24 h or more are now relative to each target timestamp. With a 24-step forecast, even horizon 24 with lag 24 points to `origin - 1 h`, so the feature remains available at forecast time. This restores useful "same target hour yesterday/week ago" information without leakage.

**2. Correct Ridge preprocessing.** The final Ridge implementation uses the already corrected standalone design: continuous variables are standardized, `hour/day_of_week/month/horizon` are one-hot encoded, binary weekend/holiday indicators pass through, and alpha is selected on validation from the expanded grid.

**3. Horizon metrics use the same seed aggregation as the main results.** For XGBoost/LSTM/TFT, MAE and RMSE are calculated separately for each seed at each horizon and then summarized as mean ± sample SD. The main horizon table no longer evaluates an implicit seed-averaged ensemble.

**4. Stronger reproducibility and leakage audits.** The notebook exports LSTM scaler parameters, TFT information-role metadata, explicit TFT decoder-variable leakage checks, tabular feature semantics, and a horizon-vs-overall MAE consistency audit.

The LSTM remains a direct 24-output sequence benchmark using historical calendar variables in its encoder; unlike TFT, it does not receive a separate future-known-calendar decoder input. This architectural information-set difference is exported explicitly and should be acknowledged in the thesis limitations.


In [ ]:
# ============================================================
# 00 ENVIRONMENT / OPTIONAL INSTALL
# ============================================================

import sys
import subprocess
import importlib.util

# Kaggle normally includes torch/xgboost/sklearn.
# Install PyTorch Forecasting + Lightning only if missing.
missing = []
if importlib.util.find_spec("pytorch_forecasting") is None:
    missing.append("pytorch-forecasting")
if importlib.util.find_spec("lightning") is None:
    missing.append("lightning")

if missing:
    print("Installing:", missing)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )
else:
    print("Required forecasting packages already available.")

In [ ]:
# ============================================================
# 00 CONFIGURATION
# ============================================================

from pathlib import Path
import os
import json
import time
import random
import traceback
import warnings
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---------------------------
# Core experiment definition
# ---------------------------
# v3 changes the TABULAR feature semantics only:
# - latest observed state remains origin-relative (o-1)
# - daily/weekly lags are target-relative, but only when guaranteed to
#   remain strictly before the common forecast origin for all 24 horizons.
# - corrected Ridge preprocessing from the standalone Ridge rerun is used.
# Neural model architectures and information boundaries are otherwise unchanged.
PROTOCOL_VERSION = "midnight_origin_v3_safe_target_lags"

SEEDS = [42, 123, 777]
LOOKBACK_SELECTION_SEEDS = list(SEEDS)

FORECAST_HORIZON = 24
FORECAST_START_LOCAL_HOUR = 0   # one forecast per local calendar day at 00:00 Europe/Berlin

LOOKBACKS = [24, 72, 168, 336]
DEFAULT_LOOKBACK = 168

# These were already selected on 2024 validation under the neural-model
# specification and do not depend on the tabular lag correction.
# Set RUN_LOOKBACK_STUDY=True if you want to reproduce the complete study.
FROZEN_LSTM_LOOKBACK = 168
FROZEN_TFT_LOOKBACK = 336

TARGET = "grid_load"

WEATHER_FEATURES = [
    "temperature_mean",
    "humidity_mean",
]

# These columns are already present in the frozen train/validation/test CSVs.
# The notebook does not recreate the holiday feature, so document its original
# source in the thesis/data-preparation code rather than guessing it here.
CALENDAR_FEATURES = [
    "hour",
    "day_of_week",
    "is_weekend",
    "month",
    "is_holiday",
]

# ---------------------------
# Leakage-safe tabular history
# ---------------------------
# The last observed system/weather state is fixed relative to the forecast
# origin (o-1) and is available for every horizon.
ORIGIN_STATE_LOAD_LAGS = [1]
ORIGIN_STATE_WEATHER_LAGS = [1]

# These lags are relative to EACH TARGET timestamp.
# With a 24-step forecast starting at the origin, lag >= 24 guarantees that
# even horizon 24 uses a source timestamp no later than o-1.
TARGET_RELATIVE_LOAD_LAGS = [24, 48, 72, 168, 336]
TARGET_RELATIVE_WEATHER_LAGS = [24, 48, 168]

TABULAR_TARGET_FEATURES = [*CALENDAR_FEATURES, "horizon"]

# Corrected Ridge setup from the standalone second Ridge run.
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
RIDGE_CATEGORICAL_FEATURES = ["hour", "day_of_week", "month", "horizon"]
RIDGE_BINARY_FEATURES = ["is_weekend", "is_holiday"]

# XGBoost
XGB_N_ESTIMATORS = 1500
XGB_LEARNING_RATE = 0.03
XGB_MAX_DEPTH = 6
XGB_EARLY_STOPPING_ROUNDS = 50

# Neural models
BATCH_SIZE = 64
SMOKE_TEST = False
SMOKE_EPOCHS = 2
LSTM_MAX_EPOCHS = 40
TFT_MAX_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 6

LSTM_HIDDEN_SIZE = 64
LSTM_NUM_LAYERS = 2
LSTM_DROPOUT = 0.2
LSTM_LEARNING_RATE = 1e-3

TFT_HIDDEN_SIZE = 32
TFT_ATTENTION_HEADS = 4
TFT_DROPOUT = 0.1
TFT_HIDDEN_CONTINUOUS_SIZE = 16
TFT_LEARNING_RATE = 1e-3

# ---------------------------
# Run switches
# ---------------------------
RUN_BASELINES = True

# The lookback study does NOT need to be repeated for the tabular lag fix.
# It can still be reproduced by setting this to True.
RUN_LOOKBACK_STUDY = False

RUN_FINAL_EXPERIMENTS = True
RUN_VISUALIZATIONS = True

# Weather ablation
WEATHER_VARIANTS = [False, True]

# Device
DEVICE = "cuda"  # automatically falls back to CPU if unavailable

# Keep TFT on one GPU by default for notebook stability.
TFT_GPU_DEVICES = 1

# ---------------------------
# Repository / output paths
# ---------------------------
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "results" / "thesis_outputs_v3"
LOG_DIR = OUTPUT_DIR / "logs"
METRICS_DIR = OUTPUT_DIR / "metrics"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
MODEL_DIR = OUTPUT_DIR / "models"
CONFIG_DIR = OUTPUT_DIR / "configs"
VIS_DIR = PROJECT_ROOT / "figures"

for folder in [
    OUTPUT_DIR, LOG_DIR, METRICS_DIR, PREDICTIONS_DIR,
    CHECKPOINT_DIR, MODEL_DIR, CONFIG_DIR, VIS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# ---------------------------
# Safety assertions
# ---------------------------
assert FORECAST_HORIZON == 24, "This notebook is designed for a 24-step forecast."

assert all(lag >= 1 for lag in ORIGIN_STATE_LOAD_LAGS)
assert all(lag >= 1 for lag in ORIGIN_STATE_WEATHER_LAGS)

# Critical leakage guarantee for target-relative features:
# horizon 24 has step=23, so lag 24 -> source o-1.
assert min(TARGET_RELATIVE_LOAD_LAGS) >= FORECAST_HORIZON
assert min(TARGET_RELATIVE_WEATHER_LAGS) >= FORECAST_HORIZON

assert LOOKBACK_SELECTION_SEEDS == SEEDS
assert set(WEATHER_VARIANTS) == {False, True}

print("Protocol:", PROTOCOL_VERSION)
print("SMOKE_TEST:", SMOKE_TEST)
print("Forecast horizon:", FORECAST_HORIZON)
print("Forecast origin local hour:", FORECAST_START_LOCAL_HOUR)
print("Origin-state load lags:", ORIGIN_STATE_LOAD_LAGS)
print("Target-relative load lags:", TARGET_RELATIVE_LOAD_LAGS)
print("Origin-state weather lags:", ORIGIN_STATE_WEATHER_LAGS)
print("Target-relative weather lags:", TARGET_RELATIVE_WEATHER_LAGS)
print("Lookbacks:", LOOKBACKS)
print("Seeds:", SEEDS)
print("Run lookback study:", RUN_LOOKBACK_STUDY)
print("Output:", OUTPUT_DIR)


In [ ]:
# ============================================================
# SHARED UTILITIES
# ============================================================

import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=str)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def log_error(experiment_name, error, metadata=None):
    path = LOG_DIR / f"{experiment_name}_ERROR.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"Experiment: {experiment_name}\n")
        f.write(f"UTC time: {datetime.now(timezone.utc).isoformat()}\n")
        if metadata:
            f.write(f"Metadata: {json.dumps(metadata, default=str)}\n")
        f.write(f"Error: {repr(error)}\n\n")
        f.write(traceback.format_exc())
    print("Error logged:", path)


def resolve_input_file(filename):
    """Resolve processed splits locally first, then fall back to Kaggle inputs."""
    local_candidate = DATA_DIR / filename
    if local_candidate.exists():
        return local_candidate

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        matches = sorted(kaggle_root.rglob(filename))
        if matches:
            if len(matches) > 1:
                print(f"Multiple {filename} files found on Kaggle; using:", matches[0])
            return matches[0]

    raise FileNotFoundError(
        f"Could not find {filename}. Expected {local_candidate} or a Kaggle input "
        "dataset containing train.csv, validation.csv and test.csv."
    )


def weather_tag(include_weather):
    return "weather" if include_weather else "no_weather"


def valid_daily_target_start_indices(df, split_name, history_required, horizon):
    """
    Return first-target row indices for one forecast per local calendar day.
    The encoder/history ends immediately before target_start.
    """
    candidates = df.index[
        (df["split"] == split_name)
        & (df["local_time"].dt.hour == FORECAST_START_LOCAL_HOUR)
    ].to_numpy()

    valid = []
    history_required = int(history_required)
    horizon = int(horizon)

    for target_start in candidates:
        target_start = int(target_start)

        # Boundary safety: enough observed history and enough future target rows.
        if target_start - history_required < 0:
            continue
        if target_start + horizon > len(df):
            continue

        target_splits = df.iloc[
            target_start : target_start + horizon
        ]["split"]

        if len(target_splits) != horizon:
            continue
        if not (target_splits == split_name).all():
            continue

        valid.append(target_start)

    return np.asarray(valid, dtype=np.int64)


def prediction_frame_from_tabular(df, prediction, model, include_weather=None, seed=None):
    """Standard prediction file for daily-origin tabular models."""
    required = [
        "timestamp", TARGET, "horizon", "forecast_date",
        "forecast_origin", "latest_observation_timestamp",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Tabular prediction frame missing columns: {missing}")

    out = df[required].copy().reset_index(drop=True)
    out.rename(columns={TARGET: "actual"}, inplace=True)
    out["prediction"] = np.asarray(prediction)
    out["model"] = model
    out["weather"] = include_weather
    out["seed"] = seed
    return out[
        [
            "forecast_origin", "latest_observation_timestamp", "timestamp",
            "horizon", "forecast_date", "actual", "prediction",
            "model", "weather", "seed",
        ]
    ]


def aggregate_seed_predictions(prediction_files):
    frames = []
    for p in prediction_files:
        d = pd.read_csv(p)
        for col in ["timestamp", "forecast_origin", "latest_observation_timestamp"]:
            if col in d.columns:
                d[col] = pd.to_datetime(d[col], utc=True)
        frames.append(d)

    all_pred = pd.concat(frames, ignore_index=True)

    group_cols = ["timestamp", "horizon"]
    if "forecast_origin" in all_pred.columns:
        group_cols = ["forecast_origin", *group_cols]

    agg_spec = {
        "actual": ("actual", "first"),
        "prediction": ("prediction", "mean"),
    }
    if "forecast_date" in all_pred.columns:
        agg_spec["forecast_date"] = ("forecast_date", "first")
    if "latest_observation_timestamp" in all_pred.columns:
        agg_spec["latest_observation_timestamp"] = (
            "latest_observation_timestamp", "first"
        )

    grouped = all_pred.groupby(group_cols, as_index=False).agg(**agg_spec)
    return grouped.sort_values(group_cols).reset_index(drop=True)


In [ ]:
# Save the complete configuration before any training.
experiment_config = {
    "protocol_version": PROTOCOL_VERSION,
    "seeds": SEEDS,
    "lookback_selection_seeds": LOOKBACK_SELECTION_SEEDS,
    "forecast_horizon": FORECAST_HORIZON,
    "forecast_frequency": "once_per_local_calendar_day",
    "forecast_start_local_hour": FORECAST_START_LOCAL_HOUR,
    "lookbacks": LOOKBACKS,
    "default_lookback": DEFAULT_LOOKBACK,
    "frozen_lstm_lookback": FROZEN_LSTM_LOOKBACK,
    "frozen_tft_lookback": FROZEN_TFT_LOOKBACK,
    "target": TARGET,
    "weather_features": WEATHER_FEATURES,
    "weather_data_type": "historical ERA5-Land reanalysis values already present in the frozen CSVs",
    "calendar_features": CALENDAR_FEATURES,
    "tabular_feature_semantics": {
        "origin_state_load_lags": ORIGIN_STATE_LOAD_LAGS,
        "origin_state_weather_lags": ORIGIN_STATE_WEATHER_LAGS,
        "target_relative_load_lags": TARGET_RELATIVE_LOAD_LAGS,
        "target_relative_weather_lags": TARGET_RELATIVE_WEATHER_LAGS,
        "guarantee": (
            "Every historical source timestamp is strictly earlier than the common "
            "forecast origin. Target-relative lags are used only when lag >= forecast horizon."
        ),
    },
    "tabular_target_features": TABULAR_TARGET_FEATURES,
    "ridge_alphas": RIDGE_ALPHAS,
    "ridge_categorical_features": RIDGE_CATEGORICAL_FEATURES,
    "ridge_binary_features": RIDGE_BINARY_FEATURES,
    "xgb_n_estimators": XGB_N_ESTIMATORS,
    "xgb_learning_rate": XGB_LEARNING_RATE,
    "xgb_max_depth": XGB_MAX_DEPTH,
    "xgb_early_stopping_rounds": XGB_EARLY_STOPPING_ROUNDS,
    "batch_size": BATCH_SIZE,
    "smoke_test": SMOKE_TEST,
    "smoke_epochs": SMOKE_EPOCHS,
    "lstm_max_epochs": LSTM_MAX_EPOCHS,
    "tft_max_epochs": TFT_MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "device": DEVICE,
    "tft_gpu_devices": TFT_GPU_DEVICES,
}
save_json(experiment_config, CONFIG_DIR / "experiment_config.json")

save_json(
    {
        "note": (
            "The holiday feature is precomputed in the frozen CSVs. "
            "Its original library/source must be documented from the preprocessing pipeline."
        )
    },
    CONFIG_DIR / "holiday_feature_source_NOTE.json",
)

print("Configuration saved.")


## 01 — Data loading and checks

The notebook first reads `data/processed/train.csv`, `validation.csv`, and `test.csv` created by Notebook 02. When run as a standalone Kaggle notebook, it can also fall back to matching files under `/kaggle/input`. Timestamps are restored and the frozen chronological split is verified before any model fitting begins.


In [ ]:
# ============================================================
# 01 DATA LOAD
# ============================================================

TRAIN_PATH = resolve_input_file("train.csv")
VAL_PATH = resolve_input_file("validation.csv")
TEST_PATH = resolve_input_file("test.csv")

print("Train:", TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Test:", TEST_PATH)

train = pd.read_csv(TRAIN_PATH)
validation = pd.read_csv(VAL_PATH)
test = pd.read_csv(TEST_PATH)

for split_df in [train, validation, test]:
    split_df["timestamp"] = pd.to_datetime(split_df["timestamp"], utc=True)
    split_df["local_time"] = (
        pd.to_datetime(split_df["local_time"], utc=True)
        .dt.tz_convert("Europe/Berlin")
    )
    split_df.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

EXPECTED_COLUMNS = [
    "timestamp", "grid_load", "temperature_mean", "humidity_mean",
    "local_time", "hour", "day_of_week", "is_weekend", "month", "is_holiday",
]

for name, split_df in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    assert set(EXPECTED_COLUMNS).issubset(split_df.columns), f"Missing columns in {name}"
    assert split_df.isna().sum().sum() == 0, f"Missing values in {name}"
    assert split_df["timestamp"].duplicated().sum() == 0, f"Duplicate timestamps in {name}"
    assert split_df["timestamp"].is_monotonic_increasing, f"{name} is not sorted"
    print(
        f"{name:10s}",
        split_df.shape,
        split_df["local_time"].min(),
        "→",
        split_df["local_time"].max(),
    )

assert train["timestamp"].max() < validation["timestamp"].min()
assert validation["timestamp"].max() < test["timestamp"].min()

# Frozen local-calendar year split.
assert set(train["local_time"].dt.year.unique()) <= set(range(2015, 2024))
assert set(validation["local_time"].dt.year.unique()) == {2024}
assert set(test["local_time"].dt.year.unique()) == {2025}

print("✓ Data and chronological split validated.")

In [ ]:
# Continuous timeline used only to construct legitimate historical context.
full_data = pd.concat(
    [
        train.assign(split="train"),
        validation.assign(split="validation"),
        test.assign(split="test"),
    ],
    ignore_index=True,
).sort_values("timestamp").reset_index(drop=True)

full_data["time_idx"] = np.arange(len(full_data), dtype=np.int64)
full_data["series_id"] = "Germany"

# Hourly continuity check in UTC.
diffs = full_data["timestamp"].diff().dropna()
assert (diffs == pd.Timedelta(hours=1)).all()

# Daily forecast-window counts for the default lookback.
for split_name in ["train", "validation", "test"]:
    starts = valid_daily_target_start_indices(
        full_data, split_name, DEFAULT_LOOKBACK, FORECAST_HORIZON
    )
    print(split_name, "daily forecast windows:", len(starts))

print("✓ Continuous hourly timeline validated.")

## 02 — Seasonal naive baseline

Two deliberately simple seasonal references are compared on **validation only**:
- `t − 24 h` (same hour yesterday)
- `t − 168 h` (same hour last week)

Seasonal Naive is evaluated only on the **same once-daily midnight-origin 24-step target windows** used by LSTM/TFT and the tabular baselines. Every seasonal source timestamp is asserted to occur before the daily forecast origin. The better validation lag is then evaluated on the untouched 2025 test period. No seed is required because the method is deterministic.


In [ ]:
# ============================================================
# 02 SEASONAL NAIVE — VALIDATION SELECTION ONLY
# ============================================================


def seasonal_naive_for_split(full_df, split_name, lag):
    """
    Same-hour seasonal naive prediction, restricted to the common
    once-daily midnight-origin 24-step forecast windows.
    """
    lag = int(lag)
    assert lag >= FORECAST_HORIZON, (
        "Seasonal naive lag must keep every source row before the daily origin."
    )

    starts = valid_daily_target_start_indices(
        full_df,
        split_name,
        history_required=lag,
        horizon=FORECAST_HORIZON,
    )

    rows = []

    for origin_idx in starts:
        origin_idx = int(origin_idx)
        forecast_origin = full_df.iloc[origin_idx]["timestamp"]
        latest_observation_timestamp = full_df.iloc[origin_idx - 1]["timestamp"]
        forecast_date = str(full_df.iloc[origin_idx]["local_time"].date())

        for step in range(FORECAST_HORIZON):
            horizon = step + 1
            target_idx = origin_idx + step
            source_idx = target_idx - lag

            # Critical leakage guard: the seasonal source must already exist
            # before the common daily forecast origin.
            assert source_idx >= 0
            assert source_idx < origin_idx
            assert target_idx >= origin_idx

            rows.append({
                "forecast_origin": forecast_origin,
                "latest_observation_timestamp": latest_observation_timestamp,
                "timestamp": full_df.iloc[target_idx]["timestamp"],
                "actual": float(full_df.iloc[target_idx][TARGET]),
                "prediction": float(full_df.iloc[source_idx][TARGET]),
                "horizon": horizon,
                "forecast_date": forecast_date,
                "model": "Seasonal Naive",
                "weather": False,
                "seed": np.nan,
            })

    out = pd.DataFrame(rows)
    if len(out):
        counts = out.groupby("forecast_origin").size()
        assert counts.eq(FORECAST_HORIZON).all()
        assert set(out["horizon"].unique()) == set(range(1, FORECAST_HORIZON + 1))

    return out


BEST_SEASONAL_LAG = 168

if RUN_BASELINES:
    seasonal_val_rows = []

    for lag in [24, 168]:
        pred = seasonal_naive_for_split(full_data, "validation", lag)
        m = calculate_metrics(pred["actual"], pred["prediction"])
        seasonal_val_rows.append({"lag": lag, **m})

    seasonal_validation_results = pd.DataFrame(seasonal_val_rows)
    display(seasonal_validation_results)
    seasonal_validation_results.to_csv(
        METRICS_DIR / "seasonal_naive_validation_lags.csv", index=False
    )

    BEST_SEASONAL_LAG = int(
        seasonal_validation_results.sort_values(["MAE", "RMSE"]).iloc[0]["lag"]
    )

    save_json(
        {
            "best_lag": BEST_SEASONAL_LAG,
            "selection_metric": "validation_MAE",
            "protocol": PROTOCOL_VERSION,
        },
        CONFIG_DIR / "seasonal_naive_selection.json",
    )

    print("Selected seasonal lag:", BEST_SEASONAL_LAG)
    print("✓ Test set has NOT been evaluated here.")


## 03 — Midnight-origin tabular features + Ridge

Ridge and XGBoost now use the **same daily forecast origin as LSTM/TFT**. A forecast begins at 00:00 German local time and predicts 24 hourly targets (`horizon = 1...24`).

Historical load and weather features are computed **relative to the forecast origin**, not relative to each target row. Therefore `origin_load_lag_1` is safe: it is the final observed load immediately before the forecast starts and is reused for all 24 target rows from that origin. Weather receives the same treatment (`origin_temperature_lag_*`, `origin_humidity_lag_*`).

The 24 rows differ through their **target-hour calendar variables** and an explicit `horizon` feature. Raw future ERA5 weather is never supplied. Earliest training origins without enough history for the maximum configured lag are dropped deliberately.

Ridge uses a train-fitted `StandardScaler`. Alpha is selected on 2024 validation only.


In [ ]:
# ============================================================
# 03 DAILY-ORIGIN TABULAR FEATURE CONSTRUCTION
# ============================================================

WEATHER_PREFIX = {
    "temperature_mean": "temperature",
    "humidity_mean": "humidity",
}


def build_safe_tabular_split(full_df, split_name, include_weather):
    """
    Create 24 supervised rows for each valid daily forecast origin.

    Information design
    ------------------
    1) Origin-state features (lag 1) are fixed for all 24 horizons:
         source = forecast_origin - 1 hour

    2) Daily/weekly historical lags are target-relative:
         source = target_timestamp - lag

       They are used ONLY for lag >= FORECAST_HORIZON. Therefore, even for
       horizon 24, lag 24 points to the final observed hour before the common
       forecast origin. Every historical source is asserted to be < origin.

    3) Target-hour calendar variables and horizon are future-known and may vary
       across the 24 target rows.

    This gives Ridge/XGBoost useful "same target hour yesterday/week ago"
    information without introducing future-load or future-weather leakage.
    """
    all_history_lags = (
        list(ORIGIN_STATE_LOAD_LAGS)
        + list(TARGET_RELATIVE_LOAD_LAGS)
    )
    if include_weather:
        all_history_lags += (
            list(ORIGIN_STATE_WEATHER_LAGS)
            + list(TARGET_RELATIVE_WEATHER_LAGS)
        )

    max_required_history = max(all_history_lags)

    starts = valid_daily_target_start_indices(
        full_df,
        split_name,
        history_required=max_required_history,
        horizon=FORECAST_HORIZON,
    )

    load_features = [
        f"origin_load_lag_{lag}" for lag in ORIGIN_STATE_LOAD_LAGS
    ] + [
        f"target_load_lag_{lag}" for lag in TARGET_RELATIVE_LOAD_LAGS
    ]

    weather_features = []
    if include_weather:
        for variable in WEATHER_FEATURES:
            prefix = WEATHER_PREFIX[variable]
            weather_features.extend(
                f"origin_{prefix}_lag_{lag}"
                for lag in ORIGIN_STATE_WEATHER_LAGS
            )
            weather_features.extend(
                f"target_{prefix}_lag_{lag}"
                for lag in TARGET_RELATIVE_WEATHER_LAGS
            )

    features = load_features + TABULAR_TARGET_FEATURES.copy() + weather_features

    # Static semantic guards.
    assert min(TARGET_RELATIVE_LOAD_LAGS) >= FORECAST_HORIZON
    if include_weather:
        assert min(TARGET_RELATIVE_WEATHER_LAGS) >= FORECAST_HORIZON
    assert TARGET not in features
    assert all(raw_weather not in features for raw_weather in WEATHER_FEATURES)
    assert "horizon" in features

    rows = []

    for origin_idx in starts:
        origin_idx = int(origin_idx)
        assert origin_idx - max_required_history >= 0

        forecast_origin = full_df.iloc[origin_idx]["timestamp"]
        forecast_origin_local = full_df.iloc[origin_idx]["local_time"]
        latest_observation_idx = origin_idx - 1
        latest_observation_timestamp = full_df.iloc[
            latest_observation_idx
        ]["timestamp"]

        assert latest_observation_timestamp < forecast_origin

        # Origin-state values are fixed across all 24 targets.
        origin_state = {}

        for lag in ORIGIN_STATE_LOAD_LAGS:
            source_idx = origin_idx - int(lag)
            source_ts = full_df.iloc[source_idx]["timestamp"]
            assert source_idx < origin_idx
            assert source_ts < forecast_origin
            origin_state[f"origin_load_lag_{lag}"] = float(
                full_df.iloc[source_idx][TARGET]
            )

        if include_weather:
            for variable in WEATHER_FEATURES:
                prefix = WEATHER_PREFIX[variable]
                for lag in ORIGIN_STATE_WEATHER_LAGS:
                    source_idx = origin_idx - int(lag)
                    source_ts = full_df.iloc[source_idx]["timestamp"]
                    assert source_idx < origin_idx
                    assert source_ts < forecast_origin
                    origin_state[f"origin_{prefix}_lag_{lag}"] = float(
                        full_df.iloc[source_idx][variable]
                    )

        for step in range(FORECAST_HORIZON):
            horizon = step + 1
            target_idx = origin_idx + step

            assert target_idx >= origin_idx
            assert target_idx < origin_idx + FORECAST_HORIZON
            assert full_df.iloc[target_idx]["split"] == split_name

            target_row = full_df.iloc[target_idx]

            row = {
                **origin_state,
                "horizon": horizon,
                "timestamp": target_row["timestamp"],
                "local_time": target_row["local_time"],
                "forecast_origin": forecast_origin,
                "forecast_origin_local": forecast_origin_local,
                "latest_observation_timestamp": latest_observation_timestamp,
                "forecast_date": str(forecast_origin_local.date()),
                "origin_idx": origin_idx,
                "target_idx": target_idx,
                "split": split_name,
                TARGET: float(target_row[TARGET]),
            }

            # Leakage-safe target-relative load lags.
            for lag in TARGET_RELATIVE_LOAD_LAGS:
                source_idx = target_idx - int(lag)
                source_ts = full_df.iloc[source_idx]["timestamp"]
                assert source_idx >= 0
                assert source_idx < origin_idx, (
                    f"Leakage: target-relative load lag {lag}, "
                    f"horizon {horizon}, source_idx={source_idx}, origin_idx={origin_idx}"
                )
                assert source_ts < forecast_origin
                row[f"target_load_lag_{lag}"] = float(
                    full_df.iloc[source_idx][TARGET]
                )

            # Leakage-safe target-relative historical weather lags.
            if include_weather:
                for variable in WEATHER_FEATURES:
                    prefix = WEATHER_PREFIX[variable]
                    for lag in TARGET_RELATIVE_WEATHER_LAGS:
                        source_idx = target_idx - int(lag)
                        source_ts = full_df.iloc[source_idx]["timestamp"]
                        assert source_idx >= 0
                        assert source_idx < origin_idx, (
                            f"Leakage: target-relative weather lag {lag}, "
                            f"horizon {horizon}, source_idx={source_idx}, origin_idx={origin_idx}"
                        )
                        assert source_ts < forecast_origin
                        row[f"target_{prefix}_lag_{lag}"] = float(
                            full_df.iloc[source_idx][variable]
                        )

            for calendar_feature in CALENDAR_FEATURES:
                row[calendar_feature] = target_row[calendar_feature]

            rows.append(row)

    data = pd.DataFrame(rows)

    if len(data):
        counts = data.groupby("forecast_origin").size()
        assert counts.eq(FORECAST_HORIZON).all()

        horizon_audit = data.groupby("forecast_origin")["horizon"].agg(
            ["min", "max", "nunique"]
        )
        assert horizon_audit["min"].eq(1).all()
        assert horizon_audit["max"].eq(FORECAST_HORIZON).all()
        assert horizon_audit["nunique"].eq(FORECAST_HORIZON).all()

        assert data[features].isna().sum().sum() == 0
        assert (
            data["latest_observation_timestamp"]
            < data["forecast_origin"]
        ).all()

    return data, features


TABULAR_DATASETS = {}

for include_weather in WEATHER_VARIANTS:
    split_frames = {}
    features = None

    for split_name in ["train", "validation", "test"]:
        frame, split_features = build_safe_tabular_split(
            full_data,
            split_name,
            include_weather,
        )
        split_frames[split_name] = frame
        if features is None:
            features = split_features
        else:
            assert features == split_features

    all_tabular = pd.concat(
        [split_frames["train"], split_frames["validation"], split_frames["test"]],
        ignore_index=True,
    )

    TABULAR_DATASETS[include_weather] = {
        "data": all_tabular,
        "features": features,
        "train": split_frames["train"],
        "validation": split_frames["validation"],
        "test": split_frames["test"],
    }

    tag = weather_tag(include_weather)
    print(
        tag,
        "| features:", len(features),
        "| train rows:", len(split_frames["train"]),
        "| validation rows:", len(split_frames["validation"]),
        "| test rows:", len(split_frames["test"]),
    )
    print(features)

# Save an auditable feature list and precise feature semantics.
save_json(
    {
        weather_tag(k): v["features"]
        for k, v in TABULAR_DATASETS.items()
    },
    CONFIG_DIR / "tabular_feature_lists.json",
)

save_json(
    {
        "origin_state_load": {
            "reference": "forecast_origin",
            "lags_hours": ORIGIN_STATE_LOAD_LAGS,
        },
        "target_relative_load": {
            "reference": "target_timestamp",
            "lags_hours": TARGET_RELATIVE_LOAD_LAGS,
        },
        "origin_state_weather": {
            "reference": "forecast_origin",
            "lags_hours": ORIGIN_STATE_WEATHER_LAGS,
        },
        "target_relative_weather": {
            "reference": "target_timestamp",
            "lags_hours": TARGET_RELATIVE_WEATHER_LAGS,
        },
        "leakage_rule": (
            "All historical source timestamps must be strictly earlier than forecast_origin."
        ),
        "proof_condition": (
            "For target-relative features, minimum lag >= forecast_horizon; "
            "with H=24, horizon 24 and lag 24 maps to origin-1."
        ),
    },
    CONFIG_DIR / "tabular_feature_semantics.json",
)

print("✓ Leakage-safe tabular features constructed and audited.")


In [ ]:
# ============================================================
# 03 RIDGE — CORRECTED PREPROCESSING + VALIDATION SELECTION ONLY
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge


def make_one_hot_encoder():
    """Compatibility with older/newer scikit-learn versions."""
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop=None,
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop=None,
            sparse=False,
        )


def ridge_continuous_features(include_weather):
    features = [
        f"origin_load_lag_{lag}" for lag in ORIGIN_STATE_LOAD_LAGS
    ] + [
        f"target_load_lag_{lag}" for lag in TARGET_RELATIVE_LOAD_LAGS
    ]

    if include_weather:
        for variable in WEATHER_FEATURES:
            prefix = WEATHER_PREFIX[variable]
            features.extend(
                f"origin_{prefix}_lag_{lag}"
                for lag in ORIGIN_STATE_WEATHER_LAGS
            )
            features.extend(
                f"target_{prefix}_lag_{lag}"
                for lag in TARGET_RELATIVE_WEATHER_LAGS
            )

    return features


def make_ridge_pipeline(include_weather, alpha):
    continuous_features = ridge_continuous_features(include_weather)

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "continuous",
                StandardScaler(),
                continuous_features,
            ),
            (
                "categorical",
                make_one_hot_encoder(),
                RIDGE_CATEGORICAL_FEATURES,
            ),
            (
                "binary",
                "passthrough",
                RIDGE_BINARY_FEATURES,
            ),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("ridge", Ridge(alpha=float(alpha))),
        ]
    )


ridge_selection_rows = []

if RUN_BASELINES:
    for include_weather in WEATHER_VARIANTS:
        tag = weather_tag(include_weather)
        bundle = TABULAR_DATASETS[include_weather]
        features = bundle["features"]
        tr = bundle["train"]
        va = bundle["validation"]

        alpha_rows = []
        fitted_by_alpha = {}

        started_variant = time.perf_counter()

        for alpha in RIDGE_ALPHAS:
            model = make_ridge_pipeline(include_weather, alpha)

            fit_started = time.perf_counter()
            model.fit(tr[features], tr[TARGET])
            fit_seconds = time.perf_counter() - fit_started

            val_pred = model.predict(va[features])
            val_metrics = calculate_metrics(va[TARGET], val_pred)

            alpha_rows.append({
                "alpha": float(alpha),
                **val_metrics,
                "fit_seconds": fit_seconds,
            })
            fitted_by_alpha[float(alpha)] = model

        alpha_df = pd.DataFrame(alpha_rows).sort_values(
            ["MAE", "RMSE", "alpha"]
        ).reset_index(drop=True)

        alpha_df.to_csv(
            METRICS_DIR / f"ridge_{tag}_alpha_validation.csv",
            index=False,
        )

        best_alpha = float(alpha_df.iloc[0]["alpha"])
        best_model = fitted_by_alpha[best_alpha]
        best_val_pred = best_model.predict(va[features])
        best_val_metrics = calculate_metrics(va[TARGET], best_val_pred)

        val_pred_df = prediction_frame_from_tabular(
            va,
            best_val_pred,
            model="Ridge",
            include_weather=include_weather,
        )
        val_pred_df.to_csv(
            PREDICTIONS_DIR / f"ridge_{tag}_validation.csv",
            index=False,
        )

        joblib.dump(best_model, MODEL_DIR / f"ridge_{tag}.joblib")

        # Save transformed feature names and coefficients for auditability.
        transformed_names = (
            best_model.named_steps["preprocessor"].get_feature_names_out()
        )
        coefficients = best_model.named_steps["ridge"].coef_

        pd.DataFrame({
            "feature": transformed_names,
            "coefficient": coefficients,
            "abs_coefficient": np.abs(coefficients),
        }).sort_values(
            "abs_coefficient", ascending=False
        ).to_csv(
            METRICS_DIR / f"ridge_{tag}_coefficients.csv",
            index=False,
        )

        selection = {
            "model": "Ridge",
            "protocol": PROTOCOL_VERSION,
            "weather": include_weather,
            "best_alpha": best_alpha,
            "candidate_alphas": RIDGE_ALPHAS,
            "validation_MAE": best_val_metrics["MAE"],
            "validation_RMSE": best_val_metrics["RMSE"],
            "n_validation_forecasts": int(va["forecast_origin"].nunique()),
            "n_validation_rows": int(len(va)),
            "selection_seconds_total": time.perf_counter() - started_variant,
            "preprocessing": {
                "continuous": "StandardScaler fitted on training rows",
                "categorical": "OneHotEncoder",
                "binary": "passthrough",
                "categorical_features": RIDGE_CATEGORICAL_FEATURES,
                "binary_features": RIDGE_BINARY_FEATURES,
                "continuous_features": ridge_continuous_features(include_weather),
            },
        }

        if best_alpha in {min(RIDGE_ALPHAS), max(RIDGE_ALPHAS)}:
            selection["alpha_boundary_warning"] = (
                "Best alpha lies on the candidate-grid boundary."
            )
            print("WARNING:", tag, "best alpha is on grid boundary:", best_alpha)

        ridge_selection_rows.append(selection)
        save_json(selection, CONFIG_DIR / f"ridge_{tag}_selection.json")
        print(tag, selection)

    pd.DataFrame(ridge_selection_rows).to_csv(
        METRICS_DIR / "ridge_validation_summary.csv",
        index=False,
    )
    print("✓ Corrected Ridge preprocessing used; test set has NOT been evaluated here.")


## 04 — XGBoost

XGBoost uses the **same midnight-origin tabular dataset as Ridge**: origin-relative historical load/weather, target-hour calendar variables, and explicit horizon `1...24`. Three seeds are used because subsampling introduces randomness. Training and validation RMSE are saved for every boosting round so learning curves can be plotted later.


In [ ]:
# ============================================================
# 04 XGBOOST — TRAIN/VALIDATION ONLY
# ============================================================

import xgboost as xgb
from xgboost import XGBRegressor

xgb_validation_rows = []


def run_xgb_validation(include_weather, seed):
    tag = weather_tag(include_weather)
    name = f"xgboost_{tag}_seed_{seed}"
    metric_path = METRICS_DIR / f"{name}_validation.json"
    model_path = MODEL_DIR / f"{name}.json"

    if metric_path.exists() and model_path.exists():
        previous = load_json(metric_path)
        if previous.get("protocol") == PROTOCOL_VERSION:
            print("Skipping completed:", name)
            return previous
        print("Ignoring stale XGBoost artifact from another protocol:", name)

    bundle = TABULAR_DATASETS[include_weather]
    features = bundle["features"]
    tr = bundle["train"]
    va = bundle["validation"]

    n_estimators = 100 if SMOKE_TEST else XGB_N_ESTIMATORS
    early_stopping = 10 if SMOKE_TEST else XGB_EARLY_STOPPING_ROUNDS

    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=n_estimators,
        learning_rate=XGB_LEARNING_RATE,
        max_depth=XGB_MAX_DEPTH,
        min_child_weight=1,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        tree_method="hist",
        device="cpu",
        random_state=seed,
        n_jobs=-1,
        eval_metric="rmse",
        early_stopping_rounds=early_stopping,
    )

    started = time.time()
    model.fit(
        tr[features],
        tr[TARGET],
        eval_set=[
            (tr[features], tr[TARGET]),
            (va[features], va[TARGET]),
        ],
        verbose=False,
    )
    elapsed = time.time() - started

    evals = model.evals_result()
    train_hist = evals["validation_0"]["rmse"]
    val_hist = evals["validation_1"]["rmse"]
    pd.DataFrame({
        "boosting_round": np.arange(1, len(train_hist) + 1),
        "train_rmse": train_hist,
        "validation_rmse": val_hist,
    }).to_csv(
        METRICS_DIR / f"{name}_history.csv", index=False
    )

    val_pred = model.predict(va[features])
    val_metrics = calculate_metrics(va[TARGET], val_pred)

    prediction_frame_from_tabular(
        va,
        val_pred,
        model="XGBoost",
        include_weather=include_weather,
        seed=seed,
    ).to_csv(
        PREDICTIONS_DIR / f"{name}_validation.csv", index=False
    )

    model.save_model(model_path)

    result = {
        "model": "XGBoost",
        "protocol": PROTOCOL_VERSION,
        "weather": include_weather,
        "seed": seed,
        "best_iteration": int(getattr(model, "best_iteration", n_estimators - 1)),
        "validation_MAE": val_metrics["MAE"],
        "validation_RMSE": val_metrics["RMSE"],
        "training_seconds": elapsed,
        "device": "cpu",
        "feature_semantics": "origin-state lag 1 + leakage-safe target-relative lags >=24",
        "n_validation_forecasts": int(va["forecast_origin"].nunique()),
        "n_validation_rows": int(len(va)),
    }
    save_json(result, metric_path)
    return result


if RUN_BASELINES:
    xgb_seeds = [SEEDS[0]] if SMOKE_TEST else SEEDS

    for include_weather in WEATHER_VARIANTS:
        for seed in xgb_seeds:
            try:
                result = run_xgb_validation(include_weather, seed)
                xgb_validation_rows.append(result)
                print(result)
            except Exception as e:
                log_error(
                    f"xgboost_{weather_tag(include_weather)}_seed_{seed}",
                    e,
                    {
                        "model": "XGBoost",
                        "protocol": PROTOCOL_VERSION,
                        "seed": seed,
                        "weather": include_weather,
                    },
                )

    if xgb_validation_rows:
        pd.DataFrame(xgb_validation_rows).to_csv(
            METRICS_DIR / "xgboost_validation_all_seeds.csv", index=False
        )

    print("✓ Test set has NOT been evaluated here.")


## 05 — LSTM

The LSTM receives only historical observations in its encoder, ending immediately before the common daily midnight target start. Forecast targets are the same 24 hourly steps used by every other model. Validation/test encoders may legitimately use observations from immediately preceding periods because those values would genuinely be available by the forecast origin.

Scaling is fitted on the **training period only**. Prediction files explicitly store the forecast origin, latest observed timestamp, target timestamp, and horizon.


In [ ]:
# ============================================================
# 05 LSTM DATA / MODEL / TRAINING FUNCTIONS
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

if DEVICE == "cuda" and torch.cuda.is_available():
    TORCH_DEVICE = torch.device("cuda")
else:
    TORCH_DEVICE = torch.device("cpu")

print("LSTM device:", TORCH_DEVICE)

class DailyWindowDataset(Dataset):
    def __init__(self, feature_array, target_array, start_indices, lookback, horizon):
        self.feature_array = feature_array.astype(np.float32)
        self.target_array = target_array.astype(np.float32)
        self.start_indices = np.asarray(start_indices, dtype=np.int64)
        self.lookback = int(lookback)
        self.horizon = int(horizon)

    def __len__(self):
        return len(self.start_indices)

    def __getitem__(self, item):
        target_start = int(self.start_indices[item])
        x = self.feature_array[
            target_start - self.lookback : target_start
        ]
        y = self.target_array[
            target_start : target_start + self.horizon
        ]
        return torch.from_numpy(x), torch.from_numpy(y)

class LoadLSTM(nn.Module):
    def __init__(
        self,
        input_size,
        horizon,
        hidden_size=LSTM_HIDDEN_SIZE,
        num_layers=LSTM_NUM_LAYERS,
        dropout=LSTM_DROPOUT,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.output = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        sequence, _ = self.lstm(x)
        return self.output(sequence[:, -1, :])

def prepare_lstm_data(include_weather, lookback):
    features = [TARGET, *CALENDAR_FEATURES]
    if include_weather:
        features += WEATHER_FEATURES

    feature_scaler = StandardScaler()
    target_scaler = StandardScaler()

    train_mask = full_data["split"].eq("train")
    feature_scaler.fit(full_data.loc[train_mask, features])
    target_scaler.fit(full_data.loc[train_mask, [TARGET]])

    feature_array = feature_scaler.transform(full_data[features])
    target_array = target_scaler.transform(full_data[[TARGET]]).ravel()

    # Reproducibility metadata: all scaling parameters are fitted on TRAIN only.
    save_json(
        {
            "model": "LSTM",
            "weather": bool(include_weather),
            "lookback": int(lookback),
            "encoder_features": features,
            "feature_scaler": "StandardScaler",
            "feature_scaler_fitted_on": "train split only",
            "feature_mean": dict(zip(features, feature_scaler.mean_.tolist())),
            "feature_scale": dict(zip(features, feature_scaler.scale_.tolist())),
            "target_scaler": "StandardScaler",
            "target_scaler_fitted_on": "train split only",
            "target_mean": float(target_scaler.mean_[0]),
            "target_scale": float(target_scaler.scale_[0]),
            "inverse_transform_applied_before_metrics": True,
            "historical_calendar_in_encoder": True,
            "future_known_calendar_supplied_to_lstm": False,
            "future_realised_weather_supplied": False,
            "note": (
                "The LSTM is a direct 24-output sequence benchmark. "
                "It does not receive a separate future-known-calendar decoder input; "
                "this should be stated as a model-representation limitation when comparing with TFT."
            ),
        },
        CONFIG_DIR / f"lstm_{weather_tag(include_weather)}_lb{lookback}_preprocessing.json",
    )

    starts = {
        split_name: valid_daily_target_start_indices(
            full_data, split_name, lookback, FORECAST_HORIZON
        )
        for split_name in ["train", "validation", "test"]
    }

    datasets = {
        split_name: DailyWindowDataset(
            feature_array,
            target_array,
            starts[split_name],
            lookback,
            FORECAST_HORIZON,
        )
        for split_name in starts
    }
    return features, feature_scaler, target_scaler, starts, datasets

def lstm_loader(dataset, train_mode):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=train_mode,
        num_workers=0,
        pin_memory=(TORCH_DEVICE.type == "cuda"),
    )

def evaluate_lstm(model, loader, target_scaler, start_indices, include_weather, seed, split_name):
    model.eval()
    pred_scaled, actual_scaled = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(TORCH_DEVICE)
            pred = model(x).cpu().numpy()
            pred_scaled.append(pred)
            actual_scaled.append(y.numpy())

    pred_scaled = np.concatenate(pred_scaled, axis=0)
    actual_scaled = np.concatenate(actual_scaled, axis=0)

    pred = target_scaler.inverse_transform(
        pred_scaled.reshape(-1, 1)
    ).reshape(pred_scaled.shape)
    actual = target_scaler.inverse_transform(
        actual_scaled.reshape(-1, 1)
    ).reshape(actual_scaled.shape)

    rows = []
    for sample_i, start_idx in enumerate(start_indices):
        for h in range(FORECAST_HORIZON):
            rows.append({
                "forecast_origin": full_data.loc[start_idx, "timestamp"],
                "latest_observation_timestamp": full_data.loc[start_idx - 1, "timestamp"],
                "timestamp": full_data.loc[start_idx + h, "timestamp"],
                "actual": float(actual[sample_i, h]),
                "prediction": float(pred[sample_i, h]),
                "horizon": h + 1,
                "forecast_date": str(full_data.loc[start_idx, "local_time"].date()),
                "model": "LSTM",
                "weather": include_weather,
                "seed": seed,
            })

    pred_df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    metrics = calculate_metrics(pred_df["actual"], pred_df["prediction"])
    return pred_df, metrics

def run_lstm(include_weather, lookback, seed, purpose="final"):
    tag = weather_tag(include_weather)
    name = f"lstm_{purpose}_{tag}_lb{lookback}_seed{seed}"
    metric_path = METRICS_DIR / f"{name}.json"

    # For final runs, require test prediction for completion.
    completion_pred = PREDICTIONS_DIR / f"{name}_test.csv"
    if metric_path.exists() and (purpose != "final" or completion_pred.exists()):
        previous = load_json(metric_path)
        if previous.get("protocol") == PROTOCOL_VERSION:
            print("Skipping completed:", name)
            return previous
        print("Ignoring stale LSTM artifact from another protocol:", name)

    set_seed(seed)
    features, feature_scaler, target_scaler, starts, datasets = prepare_lstm_data(
        include_weather, lookback
    )

    train_loader = lstm_loader(datasets["train"], True)
    val_loader = lstm_loader(datasets["validation"], False)

    # Shape assertion before training.
    xb, yb = next(iter(train_loader))
    assert xb.shape[1] == lookback
    assert xb.shape[2] == len(features)
    assert yb.shape[1] == FORECAST_HORIZON

    model = LoadLSTM(
        input_size=len(features),
        horizon=FORECAST_HORIZON,
    ).to(TORCH_DEVICE)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_LEARNING_RATE)

    max_epochs = SMOKE_EPOCHS if SMOKE_TEST else LSTM_MAX_EPOCHS
    best_val = float("inf")
    no_improve = 0
    history = []

    ckpt_path = CHECKPOINT_DIR / f"{name}.pt"
    scaler_path = MODEL_DIR / f"{name}_scalers.joblib"

    started = time.time()

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_sum = 0.0
        train_n = 0

        for x, y in train_loader:
            x = x.to(TORCH_DEVICE)
            y = y.to(TORCH_DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            train_sum += loss.item() * x.size(0)
            train_n += x.size(0)

        train_loss = train_sum / train_n

        model.eval()
        val_sum = 0.0
        val_n = 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(TORCH_DEVICE)
                y = y.to(TORCH_DEVICE)
                loss = criterion(model(x), y)
                val_sum += loss.item() * x.size(0)
                val_n += x.size(0)
        val_loss = val_sum / val_n

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": val_loss,
        })
        print(
            f"{name} | epoch {epoch:03d} | "
            f"train {train_loss:.6f} | val {val_loss:.6f}"
        )

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
            joblib.dump(
                {"feature_scaler": feature_scaler, "target_scaler": target_scaler, "features": features},
                scaler_path,
            )
        else:
            no_improve += 1
            if not SMOKE_TEST and no_improve >= EARLY_STOPPING_PATIENCE:
                print("Early stopping:", name)
                break

    training_seconds = time.time() - started
    pd.DataFrame(history).to_csv(
        METRICS_DIR / f"{name}_history.csv", index=False
    )

    model.load_state_dict(torch.load(ckpt_path, map_location=TORCH_DEVICE))

    val_pred_df, val_metrics = evaluate_lstm(
        model,
        val_loader,
        target_scaler,
        starts["validation"],
        include_weather,
        seed,
        "validation",
    )
    val_pred_df.to_csv(
        PREDICTIONS_DIR / f"{name}_validation.csv", index=False
    )

    result = {
        "model": "LSTM",
        "protocol": PROTOCOL_VERSION,
        "purpose": purpose,
        "weather": include_weather,
        "lookback": lookback,
        "seed": seed,
        "epochs_completed": len(history),
        "best_validation_loss_scaled": best_val,
        "validation_MAE": val_metrics["MAE"],
        "validation_RMSE": val_metrics["RMSE"],
        "training_seconds": training_seconds,
        "device": str(TORCH_DEVICE),
        "feature_scaler_fitted_on": "train split only",
        "target_scaler_fitted_on": "train split only",
        "future_known_calendar_supplied": False,
    }

    if purpose == "final":
        test_loader = lstm_loader(datasets["test"], False)
        test_pred_df, test_metrics = evaluate_lstm(
            model,
            test_loader,
            target_scaler,
            starts["test"],
            include_weather,
            seed,
            "test",
        )
        test_pred_df.to_csv(completion_pred, index=False)
        result["test_MAE"] = test_metrics["MAE"]
        result["test_RMSE"] = test_metrics["RMSE"]

    save_json(result, metric_path)
    return result

## 06 — Temporal Fusion Transformer (TFT)

TFT uses PyTorch Forecasting's explicit distinction between **known future** variables and **unknown observed** variables:

**Known future:** calendar variables + `time_idx`  
**Unknown/encoder-observed:** target grid load and, in the weather variant, historical temperature/humidity.

Training, validation, and test datasets are filtered to the **same once-daily midnight forecast origins** used by Seasonal Naive, Ridge, XGBoost, and LSTM. Prediction files explicitly store the common forecast origin and horizon `1...24`.


In [ ]:
# ============================================================
# 06 TFT DATA / MODEL / TRAINING FUNCTIONS
# ============================================================

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

def prepare_tft_frame():
    data = full_data.copy()
    tft_cats = CALENDAR_FEATURES.copy()
    for col in tft_cats:
        data[col] = data[col].astype(str)
    return data

TFT_FRAME = prepare_tft_frame()

def build_tft_datasets(include_weather, lookback):
    unknown_reals = [TARGET]
    if include_weather:
        unknown_reals += WEATHER_FEATURES

    known_categoricals = CALENDAR_FEATURES.copy()
    known_reals = ["time_idx"]

    train_only = TFT_FRAME[TFT_FRAME["split"] == "train"].copy()

    training = TimeSeriesDataSet(
        train_only,
        time_idx="time_idx",
        target=TARGET,
        group_ids=["series_id"],
        min_encoder_length=lookback,
        max_encoder_length=lookback,
        min_prediction_length=FORECAST_HORIZON,
        max_prediction_length=FORECAST_HORIZON,
        time_varying_known_categoricals=known_categoricals,
        time_varying_known_reals=known_reals,
        time_varying_unknown_reals=unknown_reals,
        target_normalizer=GroupNormalizer(groups=["series_id"]),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=False,
        randomize_length=False,
    )

    # Filter training windows to once-daily target starts.
    train_starts = set(
        TFT_FRAME.loc[
            valid_daily_target_start_indices(
                full_data, "train", lookback, FORECAST_HORIZON
            ),
            "time_idx",
        ].astype(int).tolist()
    )
    training = training.filter(
        lambda idx: idx["time_idx_first_prediction"].isin(train_starts)
    )

    val_start_time_idx = int(
        TFT_FRAME.loc[TFT_FRAME["split"] == "validation", "time_idx"].min()
    )
    validation_ds = TimeSeriesDataSet.from_dataset(
        training,
        TFT_FRAME,
        min_prediction_idx=val_start_time_idx,
        stop_randomization=True,
    )
    valid_val_starts = set(
        TFT_FRAME.loc[
            valid_daily_target_start_indices(
                full_data, "validation", lookback, FORECAST_HORIZON
            ),
            "time_idx",
        ].astype(int).tolist()
    )
    validation_ds = validation_ds.filter(
        lambda idx: idx["time_idx_first_prediction"].isin(valid_val_starts)
    )

    test_start_time_idx = int(
        TFT_FRAME.loc[TFT_FRAME["split"] == "test", "time_idx"].min()
    )
    test_ds = TimeSeriesDataSet.from_dataset(
        training,
        TFT_FRAME,
        min_prediction_idx=test_start_time_idx,
        stop_randomization=True,
    )
    valid_test_starts = set(
        TFT_FRAME.loc[
            valid_daily_target_start_indices(
                full_data, "test", lookback, FORECAST_HORIZON
            ),
            "time_idx",
        ].astype(int).tolist()
    )
    test_ds = test_ds.filter(
        lambda idx: idx["time_idx_first_prediction"].isin(valid_test_starts)
    )

    # Reproducibility / information-set metadata.
    save_json(
        {
            "model": "TFT",
            "weather": bool(include_weather),
            "lookback": int(lookback),
            "target": TARGET,
            "target_normalizer": "GroupNormalizer(groups=['series_id'])",
            "target_normalizer_fitted_via_training_dataset": True,
            "time_varying_unknown_reals_encoder": unknown_reals,
            "time_varying_known_categoricals": known_categoricals,
            "time_varying_known_reals": known_reals,
            "future_realised_weather_supplied": False,
            "future_known_calendar_supplied": True,
            "add_relative_time_idx": True,
            "add_target_scales": True,
            "add_encoder_length": True,
            "training_rows_source": "train split only",
        },
        CONFIG_DIR / f"tft_{weather_tag(include_weather)}_lb{lookback}_preprocessing.json",
    )

    return training, validation_ds, test_ds

def tft_loader(dataset, train_mode):
    return dataset.to_dataloader(
        train=train_mode,
        batch_size=BATCH_SIZE,
        num_workers=0,
    )

def clean_lightning_history(metrics_csv):
    raw = pd.read_csv(metrics_csv)
    rows = []
    for epoch, group in raw.dropna(subset=["epoch"]).groupby("epoch"):
        train_value = np.nan
        val_value = np.nan

        if "train_loss_epoch" in group:
            values = group["train_loss_epoch"].dropna()
            if len(values):
                train_value = float(values.iloc[-1])
        if np.isnan(train_value) and "train_loss_step" in group:
            values = group["train_loss_step"].dropna()
            if len(values):
                train_value = float(values.mean())
        if np.isnan(train_value) and "train_loss" in group:
            values = group["train_loss"].dropna()
            if len(values):
                train_value = float(values.mean())

        if "val_loss" in group:
            values = group["val_loss"].dropna()
            if len(values):
                val_value = float(values.iloc[-1])

        rows.append({
            "epoch": int(epoch) + 1,
            "train_loss": train_value,
            "validation_loss": val_value,
        })

    return pd.DataFrame(rows)

def tft_prediction_dataframe(model, dataset, loader, include_weather, seed):
    predict_trainer_kwargs = (
        {"accelerator": "gpu", "devices": 1}
        if DEVICE == "cuda" and torch.cuda.is_available()
        else {"accelerator": "cpu", "devices": 1}
    )

    prediction_obj = model.predict(
        loader,
        mode="prediction",
        return_index=True,
        return_y=True,
        trainer_kwargs=predict_trainer_kwargs,
    )

    predictions = prediction_obj.output
    if hasattr(predictions, "detach"):
        predictions = predictions.detach().cpu().numpy()
    else:
        predictions = np.asarray(predictions)

    if predictions.ndim == 3:
        predictions = predictions[:, :, predictions.shape[-1] // 2]

    y_obj = prediction_obj.y
    if isinstance(y_obj, (tuple, list)):
        y_obj = y_obj[0]
    if hasattr(y_obj, "detach"):
        actual = y_obj.detach().cpu().numpy()
    else:
        actual = np.asarray(y_obj)

    index_df = prediction_obj.index.reset_index(drop=True)
    if "time_idx" not in index_df.columns:
        raise RuntimeError(
            f"TFT prediction index does not contain time_idx. Columns: {list(index_df.columns)}"
        )
    starts = index_df["time_idx"].astype(int).to_numpy()

    assert len(starts) == predictions.shape[0] == actual.shape[0]

    rows = []
    for sample_i, start_time_idx in enumerate(starts):
        for h in range(FORECAST_HORIZON):
            row_idx = int(start_time_idx + h)
            rows.append({
                "forecast_origin": full_data.loc[start_time_idx, "timestamp"],
                "latest_observation_timestamp": full_data.loc[start_time_idx - 1, "timestamp"],
                "timestamp": full_data.loc[row_idx, "timestamp"],
                "actual": float(actual[sample_i, h]),
                "prediction": float(predictions[sample_i, h]),
                "horizon": h + 1,
                "forecast_date": str(full_data.loc[start_time_idx, "local_time"].date()),
                "model": "TFT",
                "weather": include_weather,
                "seed": seed,
            })
    return pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
def run_tft(include_weather, lookback, seed, purpose="final"):
    tag = weather_tag(include_weather)
    name = f"tft_{purpose}_{tag}_lb{lookback}_seed{seed}"
    metric_path = METRICS_DIR / f"{name}.json"
    completion_pred = PREDICTIONS_DIR / f"{name}_test.csv"

    if metric_path.exists() and (purpose != "final" or completion_pred.exists()):
        previous = load_json(metric_path)
        if previous.get("protocol") == PROTOCOL_VERSION:
            print("Skipping completed:", name)
            return previous
        print("Ignoring stale TFT artifact from another protocol:", name)

    set_seed(seed)
    pl.seed_everything(seed, workers=True)

    training_ds, validation_ds, test_ds = build_tft_datasets(
        include_weather, lookback
    )
    train_loader = tft_loader(training_ds, True)
    val_loader = tft_loader(validation_ds, False)

    # Shape check before expensive training.
    xb, yb = next(iter(train_loader))
    assert xb["encoder_target"].shape[1] == lookback
    assert xb["decoder_target"].shape[1] == FORECAST_HORIZON

    ckpt_dir = CHECKPOINT_DIR / name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    checkpoint = ModelCheckpoint(
        dirpath=ckpt_dir,
        filename="best-{epoch:02d}-{val_loss:.4f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        save_last=False,
    )

    callbacks = [checkpoint]
    if not SMOKE_TEST:
        callbacks.append(
            EarlyStopping(
                monitor="val_loss",
                min_delta=1e-4,
                patience=EARLY_STOPPING_PATIENCE,
                mode="min",
                verbose= False,
            )
        )

    logger = CSVLogger(
        save_dir=str(LOG_DIR / "lightning"),
        name=name,
    )

    use_gpu = DEVICE == "cuda" and torch.cuda.is_available()
    if use_gpu:
        accelerator = "gpu"
        available = torch.cuda.device_count()
        devices = min(max(1, TFT_GPU_DEVICES), available)
        strategy = "ddp_notebook" if devices > 1 else "auto"
    else:
        accelerator = "cpu"
        devices = 1
        strategy = "auto"

    max_epochs = SMOKE_EPOCHS if SMOKE_TEST else TFT_MAX_EPOCHS

    trainer = pl.Trainer(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=devices,
    strategy=strategy,
    gradient_clip_val=0.1,
    callbacks=callbacks,
    logger=logger,
    enable_progress_bar=False,
    enable_model_summary=False,
    log_every_n_steps=50,
        
    )

    model = TemporalFusionTransformer.from_dataset(
        training_ds,
        learning_rate=TFT_LEARNING_RATE,
        hidden_size=TFT_HIDDEN_SIZE,
        attention_head_size=TFT_ATTENTION_HEADS,
        dropout=TFT_DROPOUT,
        hidden_continuous_size=TFT_HIDDEN_CONTINUOUS_SIZE,
        loss=QuantileLoss(),
        optimizer="adam",
        reduce_on_plateau_patience=4,
        log_interval=-1,
    )

    # Explicit leakage audit of TFT decoder variables. Historical weather and
    # the target may appear in the encoder, but must NOT be decoder inputs.
    decoder_reals = set(getattr(model.hparams, "time_varying_reals_decoder", []))
    decoder_cats = set(getattr(model.hparams, "time_varying_categoricals_decoder", []))
    assert TARGET not in decoder_reals, (
        f"Target unexpectedly present in TFT decoder real inputs: {decoder_reals}"
    )
    assert not (set(WEATHER_FEATURES) & decoder_reals), (
        f"Historical weather unexpectedly present in TFT decoder real inputs: {decoder_reals}"
    )
    assert set(CALENDAR_FEATURES).issubset(decoder_cats), (
        f"Expected future-known calendar inputs missing from TFT decoder: {decoder_cats}"
    )

    save_json(
        {
            "model": "TFT",
            "weather": bool(include_weather),
            "lookback": int(lookback),
            "decoder_reals": sorted(decoder_reals),
            "decoder_categoricals": sorted(decoder_cats),
            "target_in_decoder_reals": TARGET in decoder_reals,
            "weather_in_decoder_reals": sorted(set(WEATHER_FEATURES) & decoder_reals),
            "leakage_audit_passed": True,
        },
        CONFIG_DIR / f"{name}_decoder_information_audit.json",
    )

    print(
        name,
        "| parameters:",
        f"{model.size()/1e3:.1f}k",
        "| devices:",
        devices,
    )

    started = time.time()
    trainer.fit(
        model,
        train_dataloaders=train_loader,
        val_dataloaders=val_loader,
    )
    training_seconds = time.time() - started

    best_path = checkpoint.best_model_path
    if not best_path:
        raise RuntimeError(f"No TFT checkpoint was saved for {name}")

    best_model = TemporalFusionTransformer.load_from_checkpoint(best_path)

    # Save cleaned epoch history for plotting.
    lightning_metrics = Path(logger.log_dir) / "metrics.csv"
    if lightning_metrics.exists():
        history = clean_lightning_history(lightning_metrics)
        history.to_csv(METRICS_DIR / f"{name}_history.csv", index=False)

    val_loader_eval = tft_loader(validation_ds, False)
    val_pred_df = tft_prediction_dataframe(
        best_model, validation_ds, val_loader_eval, include_weather, seed
    )
    val_pred_df.to_csv(
        PREDICTIONS_DIR / f"{name}_validation.csv", index=False
    )
    val_metrics = calculate_metrics(
        val_pred_df["actual"], val_pred_df["prediction"]
    )

    result = {
        "model": "TFT",
        "protocol": PROTOCOL_VERSION,
        "purpose": purpose,
        "weather": include_weather,
        "lookback": lookback,
        "seed": seed,
        "validation_MAE": val_metrics["MAE"],
        "validation_RMSE": val_metrics["RMSE"],
        "training_seconds": training_seconds,
        "checkpoint": best_path,
        "accelerator": accelerator,
        "gpu_devices": devices,
        "training_loss": "QuantileLoss",
        "point_forecast": "central quantile when quantile dimension is present",
        "future_known_calendar_supplied": True,
        "future_realised_weather_supplied": False,
    }

    if purpose == "final":
        test_loader_eval = tft_loader(test_ds, False)
        test_pred_df = tft_prediction_dataframe(
            best_model, test_ds, test_loader_eval, include_weather, seed
        )
        test_pred_df.to_csv(completion_pred, index=False)
        test_metrics = calculate_metrics(
            test_pred_df["actual"], test_pred_df["prediction"]
        )
        result["test_MAE"] = test_metrics["MAE"]
        result["test_RMSE"] = test_metrics["RMSE"]

    save_json(result, metric_path)
    return result

## 07 — Lookback study

Lookback selection uses **all three seeds: 42, 123, and 777**, and **2024 validation only**.

For each candidate lookback (`24`, `72`, `168`, `336` hours), LSTM and TFT are trained once per seed. The notebook saves every individual seed result plus mean and standard deviation of validation MAE/RMSE and mean training time.

The selected lookback for each architecture is the one with the **lowest mean validation MAE across the three seeds**. RMSE is retained as a secondary validation metric.

The study is run on the **with-weather** version of each neural architecture. The selected lookback is then held fixed for both with-weather and no-weather final runs, making the final weather comparison a controlled ablation with the temporal context held constant.

The **2025 test set is not used for lookback selection**.

If runtime becomes unacceptable, the predefined fallback is to change only `LOOKBACKS` in Section 00 to `[168, 336]`; the three-seed rule remains unchanged.


In [ ]:
# ============================================================
# 07A LOOKBACK STUDY — SETUP / HELPERS
# ============================================================

BEST_LSTM_LOOKBACK = DEFAULT_LOOKBACK
BEST_TFT_LOOKBACK = DEFAULT_LOOKBACK


def run_lookback_job(model_name, lookback, seed):
    """
    Run exactly one lookback/seed experiment.

    Each successful run is already saved by run_lstm/run_tft
    to its own metrics JSON file, so completed experiments can
    be resumed/skipped safely.
    """

    model_name = model_name.lower()

    print(
        f"\nSTART {model_name.upper()} "
        f"| lookback={lookback} | seed={seed}",
        flush=True,
    )

    try:
        if model_name == "lstm":
            result = run_lstm(
                True,
                lookback,
                seed,
                purpose="lookback",
            )

        elif model_name == "tft":
            result = run_tft(
                True,
                lookback,
                seed,
                purpose="lookback",
            )

        else:
            raise ValueError(
                f"Unknown model_name: {model_name}"
            )

        print(
            f"DONE {model_name.upper()} "
            f"| lookback={lookback} | seed={seed}",
            flush=True,
        )

        return result

    except Exception as e:
        log_error(
            f"{model_name}_lookback_weather_lb{lookback}_seed{seed}",
            e,
            {
                "model": model_name.upper(),
                "purpose": "lookback",
                "weather": True,
                "lookback": lookback,
                "seed": seed,
            },
        )

        # Important:
        # stop this cell so Kaggle clearly marks the failed job.
        raise


def collect_lookback_results(model_name):
    """
    Reconstruct the lookback study from the individual
    per-run JSON metric files saved by run_lstm/run_tft.

    This means results do not depend on an in-memory list.
    """

    rows = []

    for lookback in LOOKBACKS:
        for seed in LOOKBACK_SELECTION_SEEDS:

            metric_path = (
                METRICS_DIR
                / f"{model_name}_lookback_weather_lb{lookback}_seed{seed}.json"
            )

            if not metric_path.exists():
                continue

            result = load_json(metric_path)

            # Do not accept artifacts from an older protocol.
            if result.get("protocol") != PROTOCOL_VERSION:
                print(
                    "Ignoring stale result:",
                    metric_path.name,
                )
                continue

            # Defensive consistency checks.
            if int(result["lookback"]) != int(lookback):
                raise AssertionError(
                    f"Lookback mismatch in {metric_path.name}"
                )

            if int(result["seed"]) != int(seed):
                raise AssertionError(
                    f"Seed mismatch in {metric_path.name}"
                )

            rows.append(result)

    return rows


def summarize_lookback_results(rows, model_name):

    if not rows:
        return None, None

    raw_df = (
        pd.DataFrame(rows)
        .drop_duplicates(
            subset=["lookback", "seed"],
            keep="last",
        )
        .sort_values(
            ["lookback", "seed"]
        )
        .reset_index(drop=True)
    )

    raw_df.to_csv(
        METRICS_DIR
        / f"{model_name}_lookback_validation_by_seed.csv",
        index=False,
    )

    summary_df = (
        raw_df
        .groupby(
            "lookback",
            as_index=False,
        )
        .agg(
            n_seeds=(
                "seed",
                "count",
            ),
            validation_MAE_mean=(
                "validation_MAE",
                "mean",
            ),
            validation_MAE_std=(
                "validation_MAE",
                "std",
            ),
            validation_RMSE_mean=(
                "validation_RMSE",
                "mean",
            ),
            validation_RMSE_std=(
                "validation_RMSE",
                "std",
            ),
            training_seconds_mean=(
                "training_seconds",
                "mean",
            ),
        )
        .sort_values("lookback")
        .reset_index(drop=True)
    )

    summary_df.to_csv(
        METRICS_DIR
        / f"{model_name}_lookback_validation_summary.csv",
        index=False,
    )

    return raw_df, summary_df


print("✓ Lookback-study helper functions ready.")
print("Lookbacks:", LOOKBACKS)
print("Seeds:", LOOKBACK_SELECTION_SEEDS)

# 24


In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 24, 42)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 24, 42)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 24, 123)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 24, 123)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 24, 777)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 24, 777)

# 72

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 72, 42)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 72, 42)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 72, 123)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 72, 123)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 72, 777)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 72, 777)

# 168

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 168, 42)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 168, 42)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 168, 123)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 168, 123)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 168, 777)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 168, 777)

# 336

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 336, 42)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 336, 42)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 336, 123)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 336, 123)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("lstm", 336, 777)

In [ ]:
if RUN_LOOKBACK_STUDY:
    run_lookback_job("tft", 336, 777)

In [ ]:
# ============================================================
# 07 FINALIZE LOOKBACK STUDY
# ============================================================

if RUN_LOOKBACK_STUDY:
    # Reconstruct results from saved per-run JSON files.
    lstm_lookback_rows = collect_lookback_results("lstm")
    tft_lookback_rows = collect_lookback_results("tft")

    print("Found LSTM lookback results:", len(lstm_lookback_rows))
    print("Found TFT lookback results:", len(tft_lookback_rows))

    expected_runs = len(LOOKBACKS) * len(LOOKBACK_SELECTION_SEEDS)
    print("Expected runs per architecture:", expected_runs)

    lstm_lb_raw, lstm_lb_summary = summarize_lookback_results(
        lstm_lookback_rows, "lstm"
    )
    if lstm_lb_summary is None:
        raise RuntimeError("No LSTM lookback results found.")
    if len(lstm_lb_raw) != expected_runs:
        raise RuntimeError(
            f"LSTM lookback study incomplete: found {len(lstm_lb_raw)} "
            f"of {expected_runs} required runs."
        )

    for lookback in LOOKBACKS:
        observed_seeds = set(
            lstm_lb_raw.loc[
                lstm_lb_raw["lookback"] == lookback, "seed"
            ].astype(int)
        )
        expected_seeds = set(LOOKBACK_SELECTION_SEEDS)
        if observed_seeds != expected_seeds:
            raise RuntimeError(
                f"LSTM lookback {lookback} incomplete. "
                f"Expected {sorted(expected_seeds)}, found {sorted(observed_seeds)}."
            )

    BEST_LSTM_LOOKBACK = int(
        lstm_lb_summary.sort_values(
            ["validation_MAE_mean", "validation_RMSE_mean"]
        ).iloc[0]["lookback"]
    )

    tft_lb_raw, tft_lb_summary = summarize_lookback_results(
        tft_lookback_rows, "tft"
    )
    if tft_lb_summary is None:
        raise RuntimeError("No TFT lookback results found.")
    if len(tft_lb_raw) != expected_runs:
        raise RuntimeError(
            f"TFT lookback study incomplete: found {len(tft_lb_raw)} "
            f"of {expected_runs} required runs."
        )

    for lookback in LOOKBACKS:
        observed_seeds = set(
            tft_lb_raw.loc[
                tft_lb_raw["lookback"] == lookback, "seed"
            ].astype(int)
        )
        expected_seeds = set(LOOKBACK_SELECTION_SEEDS)
        if observed_seeds != expected_seeds:
            raise RuntimeError(
                f"TFT lookback {lookback} incomplete. "
                f"Expected {sorted(expected_seeds)}, found {sorted(observed_seeds)}."
            )

    BEST_TFT_LOOKBACK = int(
        tft_lb_summary.sort_values(
            ["validation_MAE_mean", "validation_RMSE_mean"]
        ).iloc[0]["lookback"]
    )

    print("\nLSTM lookback summary:")
    display(lstm_lb_summary)
    print("Selected LSTM lookback:", BEST_LSTM_LOOKBACK)

    print("\nTFT lookback summary:")
    display(tft_lb_summary)
    print("Selected TFT lookback:", BEST_TFT_LOOKBACK)

    selection_source = "recomputed from 2024 validation in this run"
    expected_runs_saved = expected_runs

else:
    # The tabular lag correction does not alter the neural architectures or
    # their validation study. Reuse the already-selected 2024 lookbacks.
    BEST_LSTM_LOOKBACK = int(FROZEN_LSTM_LOOKBACK)
    BEST_TFT_LOOKBACK = int(FROZEN_TFT_LOOKBACK)
    selection_source = (
        "frozen from the previously completed 2024 validation lookback study"
    )
    expected_runs_saved = len(LOOKBACKS) * len(LOOKBACK_SELECTION_SEEDS)

    print("Lookback study skipped in this rerun.")
    print("Reusing validation-selected LSTM lookback:", BEST_LSTM_LOOKBACK)
    print("Reusing validation-selected TFT lookback:", BEST_TFT_LOOKBACK)

save_json(
    {
        "protocol": PROTOCOL_VERSION,
        "LSTM": BEST_LSTM_LOOKBACK,
        "TFT": BEST_TFT_LOOKBACK,
        "selection_seeds": LOOKBACK_SELECTION_SEEDS,
        "lookbacks_tested": LOOKBACKS,
        "expected_runs_per_model": expected_runs_saved,
        "primary_criterion": "mean_validation_MAE_across_seeds",
        "secondary_metric": "mean_validation_RMSE_across_seeds",
        "test_used_for_selection": False,
        "selection_source": selection_source,
        "lookback_study_rerun_in_this_notebook_execution": bool(RUN_LOOKBACK_STUDY),
    },
    CONFIG_DIR / "selected_lookbacks.json",
)

print("\n✓ Final lookbacks fixed before 2025 test evaluation.")
print("✓ LSTM:", BEST_LSTM_LOOKBACK)
print("✓ TFT:", BEST_TFT_LOOKBACK)
print("✓ 2025 test set was not used for lookback selection.")


## 08 — Final experiments and first test-set evaluation

All configuration/model-selection decisions have now been frozen using training + validation only. **This is the first section that evaluates the 2025 test set.**

Every model is evaluated on the **same once-daily midnight forecast origins and horizon labels 1...24**. Seasonal Naive, Ridge, and XGBoost reuse their validation-selected configurations/models. LSTM and TFT run their final three-seed weather/no-weather experiments using the selected architecture-specific lookback. Every completed run saves immediately.


In [ ]:
# ============================================================
# 08 FINAL TEST EVALUATION + FINAL LSTM/TFT EXPERIMENTS
# ============================================================

# At this point all model-selection decisions are frozen:
# - Seasonal lag selected on validation
# - Ridge alpha selected on validation
# - XGBoost early-stopping iteration selected on validation
# - LSTM/TFT lookbacks selected on validation
#
# Only now is the 2025 test set evaluated.

# ------------------------------------------------------------
# Seasonal Naive — final test
# ------------------------------------------------------------
if RUN_FINAL_EXPERIMENTS:
    seasonal_test_pred = seasonal_naive_for_split(
        full_data, "test", BEST_SEASONAL_LAG
    )
    seasonal_test_metrics = calculate_metrics(
        seasonal_test_pred["actual"], seasonal_test_pred["prediction"]
    )
    seasonal_test_pred.to_csv(
        PREDICTIONS_DIR / "seasonal_naive_test.csv", index=False
    )
    save_json(
        {
            "model": "Seasonal Naive",
            "protocol": PROTOCOL_VERSION,
            "best_validation_lag": BEST_SEASONAL_LAG,
            "test_MAE": seasonal_test_metrics["MAE"],
            "test_RMSE": seasonal_test_metrics["RMSE"],
        },
        METRICS_DIR / "seasonal_naive_test.json",
    )
    print("Seasonal Naive test:", seasonal_test_metrics)

# ------------------------------------------------------------
# Ridge — final test
# ------------------------------------------------------------
ridge_final_rows = []

if RUN_FINAL_EXPERIMENTS:
    for include_weather in WEATHER_VARIANTS:
        tag = weather_tag(include_weather)
        model_path = MODEL_DIR / f"ridge_{tag}.joblib"
        if not model_path.exists():
            print("Missing Ridge model:", model_path)
            continue

        bundle = TABULAR_DATASETS[include_weather]
        features = bundle["features"]
        te = bundle["test"]
        model = joblib.load(model_path)
        test_pred = model.predict(te[features])
        test_metrics = calculate_metrics(te[TARGET], test_pred)

        prediction_frame_from_tabular(
            te,
            test_pred,
            model="Ridge",
            include_weather=include_weather,
        ).to_csv(
            PREDICTIONS_DIR / f"ridge_{tag}_test.csv", index=False
        )

        selection = load_json(CONFIG_DIR / f"ridge_{tag}_selection.json")
        result = {
            "model": "Ridge",
            "protocol": PROTOCOL_VERSION,
            "weather": include_weather,
            "best_alpha": selection["best_alpha"],
            "test_MAE": test_metrics["MAE"],
            "test_RMSE": test_metrics["RMSE"],
        }
        ridge_final_rows.append(result)
        save_json(result, METRICS_DIR / f"ridge_{tag}_test.json")
        print(result)

    if ridge_final_rows:
        pd.DataFrame(ridge_final_rows).to_csv(
            METRICS_DIR / "ridge_test_summary.csv", index=False
        )

# ------------------------------------------------------------
# XGBoost — final test using already-fitted selected models
# ------------------------------------------------------------
xgb_final_rows = []
final_seeds = [SEEDS[0]] if SMOKE_TEST else SEEDS

if RUN_FINAL_EXPERIMENTS:
    for include_weather in WEATHER_VARIANTS:
        tag = weather_tag(include_weather)
        bundle = TABULAR_DATASETS[include_weather]
        features = bundle["features"]
        te = bundle["test"]

        for seed in final_seeds:
            name = f"xgboost_{tag}_seed_{seed}"
            model_path = MODEL_DIR / f"{name}.json"
            if not model_path.exists():
                print("Missing XGBoost model:", model_path)
                continue

            try:
                model = XGBRegressor()
                model.load_model(model_path)
                test_pred = model.predict(te[features])
                test_metrics = calculate_metrics(te[TARGET], test_pred)

                prediction_frame_from_tabular(
                    te,
                    test_pred,
                    model="XGBoost",
                    include_weather=include_weather,
                    seed=seed,
                ).to_csv(
                    PREDICTIONS_DIR / f"{name}_test.csv", index=False
                )

                validation_info = load_json(
                    METRICS_DIR / f"{name}_validation.json"
                )
                result = {
                    **validation_info,
                    "test_MAE": test_metrics["MAE"],
                    "test_RMSE": test_metrics["RMSE"],
                }
                xgb_final_rows.append(result)
                save_json(
                    result, METRICS_DIR / f"{name}_test.json"
                )
                print(result)
            except Exception as e:
                log_error(
                    f"{name}_test",
                    e,
                    {
                        "model": "XGBoost",
                        "seed": seed,
                        "weather": include_weather,
                    },
                )

    if xgb_final_rows:
        pd.DataFrame(xgb_final_rows).to_csv(
            METRICS_DIR / "xgboost_test_all_seeds.csv", index=False
        )

# ------------------------------------------------------------
# LSTM + TFT — final 3-seed runs
# ------------------------------------------------------------
lstm_final_rows = []
tft_final_rows = []

if RUN_FINAL_EXPERIMENTS:
    for include_weather in WEATHER_VARIANTS:
        for seed in final_seeds:
            try:
                result = run_lstm(
                    include_weather,
                    BEST_LSTM_LOOKBACK,
                    seed,
                    purpose="final",
                )
                lstm_final_rows.append(result)
                print(result)
            except Exception as e:
                log_error(
                    f"lstm_final_{weather_tag(include_weather)}_seed{seed}",
                    e,
                    {
                        "model": "LSTM",
                        "weather": include_weather,
                        "seed": seed,
                        "lookback": BEST_LSTM_LOOKBACK,
                    },
                )

            try:
                result = run_tft(
                    include_weather,
                    BEST_TFT_LOOKBACK,
                    seed,
                    purpose="final",
                )
                tft_final_rows.append(result)
                print(result)
            except Exception as e:
                log_error(
                    f"tft_final_{weather_tag(include_weather)}_seed{seed}",
                    e,
                    {
                        "model": "TFT",
                        "weather": include_weather,
                        "seed": seed,
                        "lookback": BEST_TFT_LOOKBACK,
                    },
                )

if lstm_final_rows:
    pd.DataFrame(lstm_final_rows).to_csv(
        METRICS_DIR / "lstm_test_all_seeds.csv", index=False
    )

if tft_final_rows:
    pd.DataFrame(tft_final_rows).to_csv(
        METRICS_DIR / "tft_test_all_seeds.csv", index=False
    )

### 08a — Strict final-experiment completion guard

Before the forecast-origin audit or final aggregation is allowed to run, this cell verifies that **every required final artifact exists**.

It requires:

- Seasonal Naive final metric + prediction;
- Ridge final metric + prediction for both weather variants;
- XGBoost all configured final seeds for both weather variants;
- LSTM all configured final seeds for both weather variants at the selected LSTM lookback;
- TFT all configured final seeds for both weather variants at the selected TFT lookback.

For the stochastic models, each JSON result must match the current protocol, weather flag, seed, and selected lookback (where applicable), and must contain final test MAE/RMSE. The corresponding test prediction CSV must also exist.

If any required run is missing or stale, the notebook stops here rather than silently producing a thesis summary from an incomplete experiment set.


In [ ]:
# ============================================================
# 08A STRICT FINAL-EXPERIMENT COMPLETION CHECK
# ============================================================

# In the full thesis run SMOKE_TEST=False, therefore final_seeds is
# [42, 123, 777]. Keeping final_seeds here also preserves the notebook's
# smoke-test behavior if it is deliberately enabled later.
EXPECTED_FINAL_SEEDS = set(int(s) for s in final_seeds)


def _require_file(path, description):
    path = Path(path)

    if not path.exists():
        raise RuntimeError(
            f"Missing required final artifact: {description}\n"
            f"Expected file: {path}"
        )

    return path


def _validate_final_metric(
    metric_path,
    prediction_path,
    model_name,
    include_weather,
    seed=None,
    expected_lookback=None,
):
    """
    Validate one completed final experiment.

    This is a completion/reproducibility guard only.
    It does not change training, predictions, model selection, or metrics.
    """

    metric_path = _require_file(
        metric_path,
        f"{model_name} final metric",
    )

    _require_file(
        prediction_path,
        f"{model_name} final prediction",
    )

    result = load_json(metric_path)

    # Reject artifacts created under an older forecasting protocol.
    if result.get("protocol") != PROTOCOL_VERSION:
        raise RuntimeError(
            f"{model_name} has a stale protocol artifact: "
            f"{metric_path.name}. "
            f"Expected {PROTOCOL_VERSION!r}, "
            f"found {result.get('protocol')!r}."
        )

    # Every final learned-model metric must identify its weather variant.
    if "weather" not in result:
        raise RuntimeError(
            f"{model_name} metric is missing the weather flag: "
            f"{metric_path.name}"
        )

    if bool(result["weather"]) != bool(include_weather):
        raise RuntimeError(
            f"{model_name} weather flag mismatch in "
            f"{metric_path.name}. "
            f"Expected {bool(include_weather)}, "
            f"found {bool(result['weather'])}."
        )

    # Stochastic models must identify and match the requested seed.
    if seed is not None:
        if "seed" not in result:
            raise RuntimeError(
                f"{model_name} metric is missing seed: "
                f"{metric_path.name}"
            )

        if int(result["seed"]) != int(seed):
            raise RuntimeError(
                f"{model_name} seed mismatch in "
                f"{metric_path.name}. "
                f"Expected {seed}, found {result['seed']}."
            )

    # LSTM/TFT must use the validation-selected architecture lookback.
    if expected_lookback is not None:
        if "lookback" not in result:
            raise RuntimeError(
                f"{model_name} metric is missing lookback: "
                f"{metric_path.name}"
            )

        if int(result["lookback"]) != int(expected_lookback):
            raise RuntimeError(
                f"{model_name} lookback mismatch in "
                f"{metric_path.name}. "
                f"Expected {expected_lookback}, "
                f"found {result['lookback']}."
            )

    # A final metric file is not considered complete without test metrics.
    for key in ["test_MAE", "test_RMSE"]:
        if key not in result:
            raise RuntimeError(
                f"{model_name} final metric is incomplete: "
                f"{metric_path.name} is missing {key}."
            )

    return result


# ------------------------------------------------------------
# 1. Deterministic final outputs
# ------------------------------------------------------------

_require_file(
    METRICS_DIR / "seasonal_naive_test.json",
    "Seasonal Naive final metric",
)
_require_file(
    PREDICTIONS_DIR / "seasonal_naive_test.csv",
    "Seasonal Naive final prediction",
)

for include_weather in WEATHER_VARIANTS:
    tag = weather_tag(include_weather)

    ridge_metric = _require_file(
        METRICS_DIR / f"ridge_{tag}_test.json",
        f"Ridge {tag} final metric",
    )
    _require_file(
        PREDICTIONS_DIR / f"ridge_{tag}_test.csv",
        f"Ridge {tag} final prediction",
    )

    ridge_result = load_json(ridge_metric)

    if ridge_result.get("protocol") != PROTOCOL_VERSION:
        raise RuntimeError(
            f"Ridge {tag} has a stale protocol artifact."
        )

    if bool(ridge_result.get("weather")) != bool(include_weather):
        raise RuntimeError(
            f"Ridge {tag} weather flag mismatch."
        )

    for key in ["test_MAE", "test_RMSE"]:
        if key not in ridge_result:
            raise RuntimeError(
                f"Ridge {tag} metric is missing {key}."
            )


# ------------------------------------------------------------
# 2. XGBoost — require every configured final seed
# ------------------------------------------------------------

for include_weather in WEATHER_VARIANTS:
    tag = weather_tag(include_weather)
    found_seeds = set()

    for seed in final_seeds:
        name = f"xgboost_{tag}_seed_{seed}"

        _validate_final_metric(
            metric_path=METRICS_DIR / f"{name}_test.json",
            prediction_path=PREDICTIONS_DIR / f"{name}_test.csv",
            model_name="XGBoost",
            include_weather=include_weather,
            seed=seed,
        )

        found_seeds.add(int(seed))

    if found_seeds != EXPECTED_FINAL_SEEDS:
        raise RuntimeError(
            f"XGBoost {tag} final seed set incomplete. "
            f"Expected {sorted(EXPECTED_FINAL_SEEDS)}, "
            f"found {sorted(found_seeds)}."
        )

    print(
        f"✓ XGBoost {tag}: all final seeds "
        f"{sorted(found_seeds)} complete."
    )


# ------------------------------------------------------------
# 3. LSTM — require every configured final seed
# ------------------------------------------------------------

for include_weather in WEATHER_VARIANTS:
    tag = weather_tag(include_weather)
    found_seeds = set()

    for seed in final_seeds:
        name = (
            f"lstm_final_{tag}_"
            f"lb{BEST_LSTM_LOOKBACK}_seed{seed}"
        )

        _validate_final_metric(
            metric_path=METRICS_DIR / f"{name}.json",
            prediction_path=PREDICTIONS_DIR / f"{name}_test.csv",
            model_name="LSTM",
            include_weather=include_weather,
            seed=seed,
            expected_lookback=BEST_LSTM_LOOKBACK,
        )

        found_seeds.add(int(seed))

    if found_seeds != EXPECTED_FINAL_SEEDS:
        raise RuntimeError(
            f"LSTM {tag} final seed set incomplete. "
            f"Expected {sorted(EXPECTED_FINAL_SEEDS)}, "
            f"found {sorted(found_seeds)}."
        )

    print(
        f"✓ LSTM {tag}: all final seeds "
        f"{sorted(found_seeds)} complete."
    )


# ------------------------------------------------------------
# 4. TFT — require every configured final seed
# ------------------------------------------------------------

for include_weather in WEATHER_VARIANTS:
    tag = weather_tag(include_weather)
    found_seeds = set()

    for seed in final_seeds:
        name = (
            f"tft_final_{tag}_"
            f"lb{BEST_TFT_LOOKBACK}_seed{seed}"
        )

        _validate_final_metric(
            metric_path=METRICS_DIR / f"{name}.json",
            prediction_path=PREDICTIONS_DIR / f"{name}_test.csv",
            model_name="TFT",
            include_weather=include_weather,
            seed=seed,
            expected_lookback=BEST_TFT_LOOKBACK,
        )

        found_seeds.add(int(seed))

    if found_seeds != EXPECTED_FINAL_SEEDS:
        raise RuntimeError(
            f"TFT {tag} final seed set incomplete. "
            f"Expected {sorted(EXPECTED_FINAL_SEEDS)}, "
            f"found {sorted(found_seeds)}."
        )

    print(
        f"✓ TFT {tag}: all final seeds "
        f"{sorted(found_seeds)} complete."
    )


print("\n✓ STRICT FINAL COMPLETION CHECK PASSED.")
print(
    "✓ Seasonal Naive and both Ridge variants are complete."
)
print(
    "✓ XGBoost, LSTM, and TFT have all required final seeds "
    "for both weather variants."
)
print(
    "✓ Safe to continue to forecast-origin audit and final aggregation."
)


### 08b — Forecast-origin alignment audit

Before aggregating final results, the notebook verifies that every available final prediction file uses the **same forecast-origin / target-timestamp / horizon keys** as the Seasonal Naive reference. It also verifies exactly 24 horizons per origin and checks that the latest observed timestamp is strictly before the forecast origin.


In [ ]:
# ============================================================
# 08B FINAL FORECAST-ORIGIN ALIGNMENT AUDIT
# ============================================================

alignment_rows = []
reference_path = PREDICTIONS_DIR / "seasonal_naive_test.csv"


def _prediction_keys(df):
    return set(
        zip(
            pd.to_datetime(df["forecast_origin"], utc=True),
            pd.to_datetime(df["timestamp"], utc=True),
            df["horizon"].astype(int),
        )
    )


def audit_prediction_file(path, model, weather, seed):
    path = Path(path)

    if not path.exists():
        alignment_rows.append({
            "model": model,
            "weather": weather,
            "seed": seed,
            "file": path.name,
            "status": "missing",
            "n_rows": 0,
            "n_origins": 0,
        })
        return False

    d = pd.read_csv(path)
    required = {
        "forecast_origin", "latest_observation_timestamp", "timestamp",
        "horizon", "actual", "prediction",
    }
    missing_columns = sorted(required - set(d.columns))
    if missing_columns:
        raise AssertionError(
            f"{path.name} is missing protocol columns: {missing_columns}"
        )

    d["forecast_origin"] = pd.to_datetime(d["forecast_origin"], utc=True)
    d["latest_observation_timestamp"] = pd.to_datetime(
        d["latest_observation_timestamp"], utc=True
    )
    d["timestamp"] = pd.to_datetime(d["timestamp"], utc=True)

    # Information cutoff check.
    assert (d["latest_observation_timestamp"] < d["forecast_origin"]).all(), path.name
    assert (d["timestamp"] >= d["forecast_origin"]).all(), path.name

    # Exactly horizons 1..24 for every origin.
    per_origin = d.groupby("forecast_origin")["horizon"].agg(
        ["count", "min", "max", "nunique"]
    )
    assert per_origin["count"].eq(FORECAST_HORIZON).all(), path.name
    assert per_origin["min"].eq(1).all(), path.name
    assert per_origin["max"].eq(FORECAST_HORIZON).all(), path.name
    assert per_origin["nunique"].eq(FORECAST_HORIZON).all(), path.name

    aligned = _prediction_keys(d) == reference_keys

    alignment_rows.append({
        "model": model,
        "weather": weather,
        "seed": seed,
        "file": path.name,
        "status": "aligned" if aligned else "KEY_MISMATCH",
        "n_rows": int(len(d)),
        "n_origins": int(d["forecast_origin"].nunique()),
    })

    return aligned


if reference_path.exists():
    reference_df = pd.read_csv(reference_path)
    reference_df["forecast_origin"] = pd.to_datetime(
        reference_df["forecast_origin"], utc=True
    )
    reference_df["latest_observation_timestamp"] = pd.to_datetime(
        reference_df["latest_observation_timestamp"], utc=True
    )
    reference_df["timestamp"] = pd.to_datetime(reference_df["timestamp"], utc=True)

    reference_keys = _prediction_keys(reference_df)

    # Audit the reference itself first.
    audit_prediction_file(reference_path, "Seasonal Naive", False, np.nan)

    for include_weather in WEATHER_VARIANTS:
        tag = weather_tag(include_weather)

        audit_prediction_file(
            PREDICTIONS_DIR / f"ridge_{tag}_test.csv",
            "Ridge",
            include_weather,
            np.nan,
        )

        for seed in final_seeds:
            audit_prediction_file(
                PREDICTIONS_DIR / f"xgboost_{tag}_seed_{seed}_test.csv",
                "XGBoost",
                include_weather,
                seed,
            )
            audit_prediction_file(
                PREDICTIONS_DIR / f"lstm_final_{tag}_lb{BEST_LSTM_LOOKBACK}_seed{seed}_test.csv",
                "LSTM",
                include_weather,
                seed,
            )
            audit_prediction_file(
                PREDICTIONS_DIR / f"tft_final_{tag}_lb{BEST_TFT_LOOKBACK}_seed{seed}_test.csv",
                "TFT",
                include_weather,
                seed,
            )

    alignment_audit = pd.DataFrame(alignment_rows)
    alignment_audit.to_csv(
        METRICS_DIR / "forecast_origin_alignment_audit.csv",
        index=False,
    )
    display(alignment_audit)

    mismatches = alignment_audit[alignment_audit["status"] == "KEY_MISMATCH"]
    assert len(mismatches) == 0, (
        "At least one model does not use the common midnight-origin forecast keys."
    )

    missing = alignment_audit[alignment_audit["status"] == "missing"]
    if len(missing):
        print("WARNING: some final prediction files are missing; check error logs.")
        display(missing[["model", "weather", "seed", "file"]])
    else:
        print("✓ All final model predictions use the same forecast origins and horizons.")
else:
    print("Forecast-origin audit skipped: seasonal_naive_test.csv does not exist yet.")


## 09 — Evaluation and aggregation

Seeded models are reported as individual runs and as mean ± standard deviation across seeds. Ridge and Seasonal Naive are deterministic.

In [ ]:
# ============================================================
# 09 AGGREGATE FINAL RESULTS
# ============================================================

summary_rows = []

def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes"}

# Seasonal Naive
seasonal_path = METRICS_DIR / "seasonal_naive_test.json"
if seasonal_path.exists():
    r = load_json(seasonal_path)
    summary_rows.append({
        "model": "Seasonal Naive",
        "weather": False,
        "n_seeds": 1,
        "MAE_mean": r["test_MAE"],
        "MAE_std": 0.0,
        "RMSE_mean": r["test_RMSE"],
        "RMSE_std": 0.0,
    })

# Ridge
ridge_path = METRICS_DIR / "ridge_test_summary.csv"
if ridge_path.exists():
    for _, r in pd.read_csv(ridge_path).iterrows():
        summary_rows.append({
            "model": "Ridge",
            "weather": as_bool(r["weather"]),
            "n_seeds": 1,
            "MAE_mean": r["test_MAE"],
            "MAE_std": 0.0,
            "RMSE_mean": r["test_RMSE"],
            "RMSE_std": 0.0,
        })

def add_seeded_summary(path, model_name):
    if not Path(path).exists():
        return
    df = pd.read_csv(path)
    if "test_MAE" not in df.columns:
        return
    for weather_value, group in df.groupby("weather"):
        summary_rows.append({
            "model": model_name,
            "weather": as_bool(weather_value),
            "n_seeds": len(group),
            "MAE_mean": group["test_MAE"].mean(),
            "MAE_std": group["test_MAE"].std(ddof=1) if len(group) > 1 else 0.0,
            "RMSE_mean": group["test_RMSE"].mean(),
            "RMSE_std": group["test_RMSE"].std(ddof=1) if len(group) > 1 else 0.0,
            "training_seconds_mean": (
                group["training_seconds"].mean()
                if "training_seconds" in group.columns
                else np.nan
            ),
        })

add_seeded_summary(METRICS_DIR / "xgboost_test_all_seeds.csv", "XGBoost")
add_seeded_summary(METRICS_DIR / "lstm_test_all_seeds.csv", "LSTM")
add_seeded_summary(METRICS_DIR / "tft_test_all_seeds.csv", "TFT")

final_summary = pd.DataFrame(summary_rows)
if len(final_summary):
    final_summary = final_summary.sort_values(
        ["model", "weather"]
    ).reset_index(drop=True)
    display(final_summary)
    final_summary.to_csv(
        METRICS_DIR / "final_model_summary.csv", index=False
    )

# Dedicated weather-ablation table for models with both variants.
if len(final_summary):
    ablation_rows = []
    for model_name, group in final_summary.groupby("model"):
        if set(group["weather"].astype(bool)) != {False, True}:
            continue
        no_weather = group[group["weather"].astype(bool) == False].iloc[0]
        with_weather = group[group["weather"].astype(bool) == True].iloc[0]
        ablation_rows.append({
            "model": model_name,
            "MAE_no_weather": no_weather["MAE_mean"],
            "MAE_weather": with_weather["MAE_mean"],
            "MAE_change_weather_minus_no_weather": (
                with_weather["MAE_mean"] - no_weather["MAE_mean"]
            ),
            "MAE_improvement_percent": (
                (no_weather["MAE_mean"] - with_weather["MAE_mean"])
                / no_weather["MAE_mean"] * 100.0
            ),
            "RMSE_no_weather": no_weather["RMSE_mean"],
            "RMSE_weather": with_weather["RMSE_mean"],
            "RMSE_change_weather_minus_no_weather": (
                with_weather["RMSE_mean"] - no_weather["RMSE_mean"]
            ),
            "RMSE_improvement_percent": (
                (no_weather["RMSE_mean"] - with_weather["RMSE_mean"])
                / no_weather["RMSE_mean"] * 100.0
            ),
        })

    weather_ablation_summary = pd.DataFrame(ablation_rows)
    if len(weather_ablation_summary):
        display(weather_ablation_summary)
        weather_ablation_summary.to_csv(
            METRICS_DIR / "weather_ablation_summary.csv", index=False
        )


## 10 — Export and thesis-ready visualizations

All thesis-ready figures are written to the repository-level `figures/` directory.

For epoch-based models, training/validation loss histories are plotted. Seasonal Naive has no training process, so no artificial loss curve is created. Ridge instead gets an alpha-vs-validation-error plot. XGBoost gets train-vs-validation RMSE by boosting round.

Actual-vs-predicted plots for stochastic models may use seed-averaged predictions purely for visualization and are labelled accordingly. **Main numerical horizon metrics are not ensemble metrics:** they are calculated per seed first and summarized as mean ± sample SD, matching the overall-results convention.


In [ ]:
# ============================================================
# 10 VISUALIZATIONS
# ============================================================

import matplotlib.pyplot as plt

def save_current_figure(filename):
    path = VIS_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved:", path)

def plot_actual_vs_pred(df, title, filename, days=7):
    d = df.copy()
    d["timestamp"] = pd.to_datetime(d["timestamp"], utc=True)
    d = d.sort_values("timestamp")
    # If two daily forecast origins overlap around a DST boundary, average the
    # forecast instances only for this readability plot. Horizon-wise metrics
    # retain the individual forecast instances.
    d = (
        d.groupby("timestamp", as_index=False)
        .agg(actual=("actual", "first"), prediction=("prediction", "mean"))
        .sort_values("timestamp")
    )
    start = d["timestamp"].min()
    end = start + pd.Timedelta(days=days)
    d = d[(d["timestamp"] >= start) & (d["timestamp"] < end)]

    plt.figure(figsize=(12, 5))
    plt.plot(d["timestamp"], d["actual"], label="Actual Grid Load")
    plt.plot(d["timestamp"], d["prediction"], label="Predicted Grid Load")
    plt.xlabel("Time")
    plt.ylabel("Hourly Grid Load [MWh]")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    save_current_figure(filename)

def plot_history_csv(path, title, filename):
    h = pd.read_csv(path)
    if not {"epoch", "train_loss", "validation_loss"}.issubset(h.columns):
        return
    h = h.dropna(subset=["train_loss", "validation_loss"], how="all")
    if len(h) == 0:
        return
    plt.figure(figsize=(8, 5))
    plt.plot(h["epoch"], h["train_loss"], label="Training Loss")
    plt.plot(h["epoch"], h["validation_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    save_current_figure(filename)

if RUN_VISUALIZATIONS:
    # -------------------------
    # Seasonal naive
    # -------------------------
    p = PREDICTIONS_DIR / "seasonal_naive_test.csv"
    if p.exists():
        plot_actual_vs_pred(
            pd.read_csv(p),
            "Seasonal Naive — Actual vs Predicted Grid Load",
            "seasonal_naive_actual_vs_predicted.png",
        )

    # -------------------------
    # Ridge
    # -------------------------
    for include_weather in WEATHER_VARIANTS:
        tag = weather_tag(include_weather)
        p = PREDICTIONS_DIR / f"ridge_{tag}_test.csv"
        if p.exists():
            plot_actual_vs_pred(
                pd.read_csv(p),
                f"Ridge ({tag.replace('_', ' ')}) — Actual vs Predicted Grid Load",
                f"ridge_{tag}_actual_vs_predicted.png",
            )

        alpha_p = METRICS_DIR / f"ridge_{tag}_alpha_validation.csv"
        if alpha_p.exists():
            a = pd.read_csv(alpha_p)
            plt.figure(figsize=(8, 5))
            plt.plot(a["alpha"], a["MAE"], marker="o", label="Validation MAE")
            plt.plot(a["alpha"], a["RMSE"], marker="o", label="Validation RMSE")
            plt.xscale("log")
            plt.xlabel("Ridge Alpha")
            plt.ylabel("Error [MWh]")
            plt.title(f"Ridge ({tag.replace('_', ' ')}) — Validation Error by Alpha")
            plt.legend()
            plt.grid(alpha=0.3)
            save_current_figure(f"ridge_{tag}_alpha_validation.png")

    # -------------------------
    # XGBoost
    # -------------------------
    for include_weather in WEATHER_VARIANTS:
        tag = weather_tag(include_weather)
        xgb_pred_files = sorted(
            PREDICTIONS_DIR.glob(f"xgboost_{tag}_seed_*_test.csv")
        )
        if xgb_pred_files:
            avg_pred = aggregate_seed_predictions(xgb_pred_files)
            plot_actual_vs_pred(
                avg_pred,
                f"XGBoost ({tag.replace('_', ' ')}) — Actual vs Seed-Averaged Predicted Grid Load",
                f"xgboost_{tag}_actual_vs_predicted.png",
            )

        for seed in final_seeds:
            history_path = METRICS_DIR / f"xgboost_{tag}_seed_{seed}_history.csv"
            if history_path.exists():
                h = pd.read_csv(history_path)
                plt.figure(figsize=(8, 5))
                plt.plot(
                    h["boosting_round"], h["train_rmse"], label="Training RMSE"
                )
                plt.plot(
                    h["boosting_round"], h["validation_rmse"], label="Validation RMSE"
                )
                plt.xlabel("Boosting Round")
                plt.ylabel("RMSE [MWh]")
                plt.title(
                    f"XGBoost ({tag.replace('_', ' ')}, seed {seed}) — "
                    "Training vs Validation RMSE"
                )
                plt.legend()
                plt.grid(alpha=0.3)
                save_current_figure(
                    f"xgboost_{tag}_seed{seed}_training_validation_rmse.png"
                )

    # -------------------------
    # LSTM and TFT
    # -------------------------
    for model_name, best_lb in [
        ("lstm", BEST_LSTM_LOOKBACK),
        ("tft", BEST_TFT_LOOKBACK),
    ]:
        display_name = model_name.upper() if model_name == "lstm" else "TFT"

        for include_weather in WEATHER_VARIANTS:
            tag = weather_tag(include_weather)

            pred_files = sorted(
                PREDICTIONS_DIR.glob(
                    f"{model_name}_final_{tag}_lb{best_lb}_seed*_test.csv"
                )
            )
            if pred_files:
                avg_pred = aggregate_seed_predictions(pred_files)
                plot_actual_vs_pred(
                    avg_pred,
                    f"{display_name} ({tag.replace('_', ' ')}) — Actual vs Seed-Averaged Predicted Grid Load",
                    f"{model_name}_{tag}_actual_vs_predicted.png",
                )

            for seed in final_seeds:
                history_path = (
                    METRICS_DIR
                    / f"{model_name}_final_{tag}_lb{best_lb}_seed{seed}_history.csv"
                )
                if history_path.exists():
                    plot_history_csv(
                        history_path,
                        (
                            f"{display_name} ({tag.replace('_', ' ')}, seed {seed}) — "
                            "Training vs Validation Loss"
                        ),
                        f"{model_name}_{tag}_seed{seed}_training_validation_loss.png",
                    )

    # -------------------------
    # Lookback validation comparison — mean ± SD across 3 seeds
    # -------------------------
    for model_name in ["lstm", "tft"]:
        lb_path = METRICS_DIR / f"{model_name}_lookback_validation_summary.csv"
        if lb_path.exists():
            d = pd.read_csv(lb_path).sort_values("lookback")
            plt.figure(figsize=(8, 5))
            plt.errorbar(
                d["lookback"], d["validation_MAE_mean"],
                yerr=d["validation_MAE_std"].fillna(0),
                marker="o", capsize=4, label="Validation MAE (mean ± SD)",
            )
            plt.errorbar(
                d["lookback"], d["validation_RMSE_mean"],
                yerr=d["validation_RMSE_std"].fillna(0),
                marker="o", capsize=4, label="Validation RMSE (mean ± SD)",
            )
            plt.xlabel("Historical Lookback [hours]")
            plt.ylabel("Error [MWh]")
            plt.title(f"{model_name.upper()} — Lookback Validation Performance (3 Seeds)")
            plt.legend()
            plt.grid(alpha=0.3)
            save_current_figure(f"{model_name}_lookback_validation_3seeds.png")

    # -------------------------
    # Final comparison
    # -------------------------
    if "final_summary" in globals() and len(final_summary):
        comparison = final_summary.copy()
        comparison["label"] = comparison.apply(
            lambda r: (
                f"{r['model']} + weather"
                if bool(r["weather"])
                else f"{r['model']} no weather"
            ),
            axis=1,
        )

        plt.figure(figsize=(11, 6))
        plt.bar(
            comparison["label"],
            comparison["MAE_mean"],
            yerr=comparison["MAE_std"].fillna(0),
            capsize=4,
        )
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("MAE [MWh]")
        plt.title("Final Test MAE — Model Comparison")
        plt.grid(axis="y", alpha=0.3)
        save_current_figure("final_model_comparison_MAE.png")

        plt.figure(figsize=(11, 6))
        plt.bar(
            comparison["label"],
            comparison["RMSE_mean"],
            yerr=comparison["RMSE_std"].fillna(0),
            capsize=4,
        )
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("RMSE [MWh]")
        plt.title("Final Test RMSE — Model Comparison")
        plt.grid(axis="y", alpha=0.3)
        save_current_figure("final_model_comparison_RMSE.png")

In [ ]:
# ============================================================
# 10 HORIZON-WISE ERROR EXPORT + VISUALIZATION
# ============================================================
# IMPORTANT:
# For stochastic models, horizon metrics are calculated PER SEED first and
# then summarized across seeds. This matches the aggregation convention used
# in the overall final-results table. We do NOT calculate the main horizon
# metrics from seed-averaged (ensemble) predictions.

horizon_seed_rows = []


def add_horizon_metrics_single_prediction_file(
    path,
    model,
    weather,
    seed_label,
):
    path = Path(path)
    if not path.exists():
        return

    d = pd.read_csv(path)

    required = {"actual", "prediction", "horizon"}
    missing = required - set(d.columns)
    if missing:
        raise ValueError(f"{path.name} missing required columns: {sorted(missing)}")

    for horizon, group in d.groupby("horizon"):
        m = calculate_metrics(group["actual"], group["prediction"])
        horizon_seed_rows.append({
            "model": model,
            "weather": bool(weather),
            "seed": seed_label,
            "horizon": int(horizon),
            "MAE": m["MAE"],
            "RMSE": m["RMSE"],
            "n_predictions": int(len(group)),
        })


# Deterministic models: one row per horizon.
add_horizon_metrics_single_prediction_file(
    PREDICTIONS_DIR / "seasonal_naive_test.csv",
    "Seasonal Naive",
    False,
    "deterministic",
)

for include_weather in WEATHER_VARIANTS:
    tag = weather_tag(include_weather)

    add_horizon_metrics_single_prediction_file(
        PREDICTIONS_DIR / f"ridge_{tag}_test.csv",
        "Ridge",
        include_weather,
        "deterministic",
    )

    # Stochastic models: keep each seed separate.
    for seed in final_seeds:
        add_horizon_metrics_single_prediction_file(
            PREDICTIONS_DIR / f"xgboost_{tag}_seed_{seed}_test.csv",
            "XGBoost",
            include_weather,
            int(seed),
        )

        add_horizon_metrics_single_prediction_file(
            PREDICTIONS_DIR
            / f"lstm_final_{tag}_lb{BEST_LSTM_LOOKBACK}_seed{seed}_test.csv",
            "LSTM",
            include_weather,
            int(seed),
        )

        add_horizon_metrics_single_prediction_file(
            PREDICTIONS_DIR
            / f"tft_final_{tag}_lb{BEST_TFT_LOOKBACK}_seed{seed}_test.csv",
            "TFT",
            include_weather,
            int(seed),
        )


if horizon_seed_rows:
    horizon_seed_df = pd.DataFrame(horizon_seed_rows).sort_values(
        ["model", "weather", "seed", "horizon"]
    )

    # Export exact seed-level values for auditability.
    horizon_seed_df.to_csv(
        METRICS_DIR / "horizon_wise_test_metrics_by_seed.csv",
        index=False,
    )

    def sample_sd(series):
        return float(series.std(ddof=1)) if len(series) > 1 else 0.0

    horizon_df = (
        horizon_seed_df
        .groupby(["model", "weather", "horizon"], as_index=False)
        .agg(
            MAE_mean=("MAE", "mean"),
            MAE_std=("MAE", sample_sd),
            RMSE_mean=("RMSE", "mean"),
            RMSE_std=("RMSE", sample_sd),
            n_seeds=("seed", "size"),
            n_predictions_per_seed_min=("n_predictions", "min"),
            n_predictions_per_seed_max=("n_predictions", "max"),
        )
        .sort_values(["model", "weather", "horizon"])
        .reset_index(drop=True)
    )

    # Every horizon should contain the same number of daily origins per seed.
    assert (
        horizon_df["n_predictions_per_seed_min"]
        == horizon_df["n_predictions_per_seed_max"]
    ).all()

    horizon_df.to_csv(
        METRICS_DIR / "horizon_wise_test_metrics.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Consistency audit against overall MAE
    # --------------------------------------------------------
    # Because each horizon has the same number of predictions, the arithmetic
    # mean of the 24 per-horizon MAEs equals the overall MAE for each seed.
    # Averaging those seed-level quantities also equals the reported overall
    # mean MAE. This catches accidental ensemble-vs-seed aggregation changes.
    consistency_rows = []

    if "final_summary" in globals() and len(final_summary):
        for (model_name, weather_value), group in horizon_df.groupby(
            ["model", "weather"]
        ):
            expected_rows = final_summary[
                (final_summary["model"] == model_name)
                & (final_summary["weather"].astype(bool) == bool(weather_value))
            ]
            if len(expected_rows) != 1:
                continue

            horizon_mean_mae = float(group["MAE_mean"].mean())
            overall_mean_mae = float(expected_rows.iloc[0]["MAE_mean"])
            diff = horizon_mean_mae - overall_mean_mae

            consistency_rows.append({
                "model": model_name,
                "weather": bool(weather_value),
                "mean_of_24_horizon_MAE_means": horizon_mean_mae,
                "overall_MAE_mean": overall_mean_mae,
                "difference": diff,
                "pass": bool(np.isclose(
                    horizon_mean_mae,
                    overall_mean_mae,
                    rtol=1e-9,
                    atol=1e-4,
                )),
            })

        horizon_consistency = pd.DataFrame(consistency_rows)
        horizon_consistency.to_csv(
            METRICS_DIR / "horizon_overall_MAE_consistency_audit.csv",
            index=False,
        )
        display(horizon_consistency)

        failed = horizon_consistency[~horizon_consistency["pass"]]
        assert len(failed) == 0, (
            "Horizon-wise MAE aggregation is inconsistent with final overall MAE."
        )

        print("✓ Horizon-wise MAE uses the same per-seed aggregation as final results.")

    # --------------------------------------------------------
    # Visualization
    # --------------------------------------------------------
    if RUN_VISUALIZATIONS:
        for metric in ["MAE", "RMSE"]:
            plt.figure(figsize=(11, 6))

            mean_col = f"{metric}_mean"
            std_col = f"{metric}_std"

            for (model_name, weather_value), group in horizon_df.groupby(
                ["model", "weather"]
            ):
                group = group.sort_values("horizon")
                label = (
                    f"{model_name} + weather"
                    if bool(weather_value)
                    else f"{model_name} no weather"
                )

                plt.plot(
                    group["horizon"],
                    group[mean_col],
                    marker="o",
                    label=label,
                )

                # Show seed variability only for stochastic models.
                if int(group["n_seeds"].max()) > 1:
                    lower = group[mean_col] - group[std_col]
                    upper = group[mean_col] + group[std_col]
                    plt.fill_between(
                        group["horizon"],
                        lower,
                        upper,
                        alpha=0.12,
                    )

            plt.xlabel("Forecast Horizon [h]")
            plt.ylabel(f"{metric} [MWh]")
            plt.title(
                f"Test {metric} by Forecast Horizon "
                "(per-seed metrics summarized as mean ± SD)"
            )
            plt.xticks(range(1, FORECAST_HORIZON + 1))
            plt.legend()
            plt.grid(alpha=0.3)
            save_current_figure(f"horizon_wise_test_{metric}.png")


In [ ]:
# ============================================================
# 10 REPRODUCIBILITY / FINAL MANIFEST
# ============================================================

# Package versions
with open(CONFIG_DIR / "package_versions.txt", "w", encoding="utf-8") as f:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=f,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )

# Runtime environment
runtime_info = {
    "protocol_version": PROTOCOL_VERSION,
    "python": sys.version,
    "platform": platform.platform(),
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_device_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
    "cuda_devices": (
        [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
        if torch.cuda.is_available()
        else []
    ),
}
save_json(runtime_info, CONFIG_DIR / "runtime_environment.json")

# File manifest
manifest = []
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        manifest.append({
            "path": str(p.relative_to(OUTPUT_DIR)),
            "bytes": p.stat().st_size,
        })

pd.DataFrame(manifest).to_csv(
    CONFIG_DIR / "output_manifest.csv", index=False
)

print("\n✓ Notebook completed.")
print("Outputs:", OUTPUT_DIR)
print("Visualizations:", VIS_DIR)
print("Files saved:", len(manifest))